
# 15 — Assembly101 Streaming ProcedureVRL + CLIP Feature Extraction — AUTOLOOP V5

This version keeps the validated V4 extraction pipeline but automatically
processes multiple streaming batches during one Colab runtime.

```text
download one batch
→ ProcedureVRL hidden features
→ CLIP visual features
→ aligned ground truth
→ validation
→ delete raw videos
→ next batch
```

The automatic loop stops when:

- all `350/350` recordings and `680/680` sequences are complete;
- `MAX_BATCHES_PER_SESSION` is reached;
- the safe runtime limit is reached;
- a batch still fails after all configured retries.

Progress is persistent in Google Drive. Rerunning this notebook continues from
the first unfinished recording.

Default configuration:

```python
STREAM_BATCH_RECORDINGS = 10
MAX_BATCHES_PER_SESSION = 100
MAX_SESSION_HOURS = 8.0
STOP_BUFFER_MINUTES = 45
MAX_RETRIES_PER_BATCH = 2
```

Raw videos are deleted only after ProcedureVRL features, CLIP features, and
aligned ground truth all pass validation.


## 1. Mount Google Drive

In [16]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Install dependencies

In [17]:
%pip install -q -U \
    "huggingface_hub[hf_xet]" \
    yacs simplejson fvcore iopath decord av einops timm \
    pandas scikit-learn opencv-python ffmpeg-python \
    pytorchvideo ipdb ftfy regex tqdm tabulate

%pip uninstall -y clip -q
%pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done


## 3. Imports

In [18]:
from pathlib import Path
from datetime import datetime, timezone
from fractions import Fraction

import hashlib
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import time
import traceback

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch

from huggingface_hub import (
    HfApi,
    get_token,
    hf_hub_download,
    login,
    notebook_login,
)
from huggingface_hub.errors import (
    GatedRepoError,
    HfHubHTTPError,
    RepositoryNotFoundError,
)

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU",
)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


## 4. Configuration

In [19]:
# ------------------------------------------------------------------
# Run mode
# ------------------------------------------------------------------
# AUTOLOOP V5 always uses persistent streaming mode.
RUN_MODE = "stream"

# Number of raw recordings per automatic batch.
STREAM_BATCH_RECORDINGS = 10
SMOKE_RECORDINGS = 2

# Automatic session controls.
AUTO_LOOP = True
MAX_BATCHES_PER_SESSION = 100
MAX_SESSION_HOURS = 8.0
STOP_BUFFER_MINUTES = 45
MAX_RETRIES_PER_BATCH = 2
DRIVE_FLUSH_SECONDS = 10
STOP_AFTER_UNRECOVERABLE_BATCH_FAILURE = True

# Smoke keeps raw videos for inspection.
# Stream deletes raw videos only after ProcedureVRL + CLIP + GT validation.
DELETE_RAW_AFTER_SUCCESS = RUN_MODE == "stream"

# ------------------------------------------------------------------
# Feature configuration
# ------------------------------------------------------------------
NUM_TEMPORAL_STEPS = 16
PROCEDUREVRL_NUM_ENSEMBLE_VIEWS = 16
PROCEDUREVRL_BATCH_SIZE = 4
PROCEDUREVRL_NUM_WORKERS = 2

EXPECTED_PROCEDUREVRL_DIM = 512

CLIP_MODEL_NAME = "ViT-B/16"
EXPECTED_CLIP_DIM = 512
CLIP_FRAME_BATCH_SIZE = 32
CLIP_NORMALIZE_FEATURES = True

TEXT_PROMPT_TEMPLATE = (
    "a video of a person performing the action: {}"
)

ANNOTATION_FPS = 30.0

# ------------------------------------------------------------------
# Download configuration
# ------------------------------------------------------------------
REPO_ID = "cvml-nus/assembly101"
REPO_TYPE = "dataset"

MAX_DOWNLOAD_ATTEMPTS = 3
RETRY_SLEEP_SECONDS = 10

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/mmf_tas_lab_data"
)
ASSEMBLY_ROOT = DRIVE_ROOT / "assembly101"

MSTCN_SOURCE_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "assembly101"
    / "coarse_mstcn_format"
)

V1_DOWNLOAD_ROOT = MSTCN_SOURCE_ROOT / "v1_download"

SOURCE_MAPPING_PATH = MSTCN_SOURCE_ROOT / "mapping.txt"
SOURCE_CLASS_METADATA_PATH = (
    MSTCN_SOURCE_ROOT / "class_metadata.csv"
)
SOURCE_DENSE_GT_DIR = MSTCN_SOURCE_ROOT / "groundTruth"
SOURCE_SPLIT_DIR = MSTCN_SOURCE_ROOT / "splits"

SEQUENCE_MANIFEST_PATH = (
    MSTCN_SOURCE_ROOT
    / "procedurevrl_extraction_manifest_v1.csv"
)
RECORDING_MANIFEST_PATH = (
    MSTCN_SOURCE_ROOT
    / "recording_download_manifest_v1.csv"
)
REMOTE_SIZE_INVENTORY_PATH = (
    V1_DOWNLOAD_ROOT
    / "remote_size_inventory_v1.csv"
)
NOTEBOOK14_SUMMARY_PATH = (
    V1_DOWNLOAD_ROOT
    / "assembly101_v1_download_summary.json"
)

PROCEDUREVRL_REPO = Path("/content/ProcedureVRL")
PROCEDUREVRL_GIT = (
    "https://github.com/facebookresearch/ProcedureVRL.git"
)
PROCEDUREVRL_CKPT_PATH = (
    DRIVE_ROOT
    / "procedurevrl"
    / "checkpoints"
    / "checkpoint_epoch_00025.pyth"
)

OUT_ROOT = (
    MSTCN_SOURCE_ROOT
    / "streaming_visual_features_v1"
)

PROC_DATASET_ROOT = (
    OUT_ROOT / "procedurevrl_hidden"
)
PROC_FEATURE_DIR = PROC_DATASET_ROOT / "features"
PROC_GT_DIR = PROC_DATASET_ROOT / "groundTruth"
PROC_SPLIT_DIR = PROC_DATASET_ROOT / "splits"

CLIP_DATASET_ROOT = (
    OUT_ROOT / "clip_vitb16"
)
CLIP_FEATURE_DIR = CLIP_DATASET_ROOT / "features"
CLIP_GT_DIR = CLIP_DATASET_ROOT / "groundTruth"
CLIP_SPLIT_DIR = CLIP_DATASET_ROOT / "splits"
CLIP_TEXT_DIR = CLIP_DATASET_ROOT / "text_embeddings"

RUNS_ROOT = OUT_ROOT / "runs"
MANIFEST_ROOT = OUT_ROOT / "manifests"
LOG_ROOT = OUT_ROOT / "logs"

SEQUENCE_STATUS_PATH = (
    MANIFEST_ROOT / "sequence_feature_status.csv"
)
RECORDING_STATUS_PATH = (
    MANIFEST_ROOT / "recording_feature_status.csv"
)
TEMPORAL_ALIGNMENT_PATH = (
    MANIFEST_ROOT / "temporal_alignment_manifest.csv"
)
FINAL_SUMMARY_PATH = (
    OUT_ROOT / "streaming_feature_extraction_summary.json"
)

for path in [
    OUT_ROOT,
    PROC_FEATURE_DIR,
    PROC_GT_DIR,
    PROC_SPLIT_DIR,
    CLIP_FEATURE_DIR,
    CLIP_GT_DIR,
    CLIP_SPLIT_DIR,
    CLIP_TEXT_DIR,
    RUNS_ROOT,
    MANIFEST_ROOT,
    LOG_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

if RUN_MODE not in {"smoke", "stream"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'stream'.")

if (
    PROCEDUREVRL_NUM_ENSEMBLE_VIEWS
    != NUM_TEMPORAL_STEPS
):
    raise ValueError(
        "ProcedureVRL views and target temporal steps must match."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "ProcedureVRL extraction should be run with a GPU runtime."
    )

print("RUN_MODE:", RUN_MODE)
print("STREAM_BATCH_RECORDINGS:", STREAM_BATCH_RECORDINGS)
print("AUTO_LOOP:", AUTO_LOOP)
print("MAX_BATCHES_PER_SESSION:", MAX_BATCHES_PER_SESSION)
print("MAX_SESSION_HOURS:", MAX_SESSION_HOURS)
print("STOP_BUFFER_MINUTES:", STOP_BUFFER_MINUTES)
print("MAX_RETRIES_PER_BATCH:", MAX_RETRIES_PER_BATCH)
print("DELETE_RAW_AFTER_SUCCESS:", DELETE_RAW_AFTER_SUCCESS)
print("OUT_ROOT:", OUT_ROOT)
print("ProcedureVRL checkpoint:", PROCEDUREVRL_CKPT_PATH)

RUN_MODE: stream
STREAM_BATCH_RECORDINGS: 10
AUTO_LOOP: True
MAX_BATCHES_PER_SESSION: 100
MAX_SESSION_HOURS: 8.0
STOP_BUFFER_MINUTES: 45
MAX_RETRIES_PER_BATCH: 2
DELETE_RAW_AFTER_SUCCESS: True
OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1
ProcedureVRL checkpoint: /content/drive/MyDrive/mmf_tas_lab_data/procedurevrl/checkpoints/checkpoint_epoch_00025.pyth


## 5. Authenticate with Hugging Face

In [20]:
def authenticate_huggingface():
    token = None

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        login(
            token=token,
            add_to_git_credential=False,
        )
        print(
            "Authenticated using Colab secret HF_TOKEN."
        )
        return

    if get_token() is not None:
        print(
            "A Hugging Face token is already available."
        )
        return

    notebook_login(skip_if_logged_in=True)


authenticate_huggingface()

if get_token() is None:
    raise RuntimeError(
        "Hugging Face authentication did not complete."
    )

Authenticated using Colab secret HF_TOKEN.


## 6. Load and validate notebook 13/14 artifacts

In [21]:
required_paths = {
    "source mapping": SOURCE_MAPPING_PATH,
    "class metadata": SOURCE_CLASS_METADATA_PATH,
    "dense ground truth": SOURCE_DENSE_GT_DIR,
    "source splits": SOURCE_SPLIT_DIR,
    "sequence manifest": SEQUENCE_MANIFEST_PATH,
    "recording manifest": RECORDING_MANIFEST_PATH,
    "remote size inventory": REMOTE_SIZE_INVENTORY_PATH,
    "notebook 14 summary": NOTEBOOK14_SUMMARY_PATH,
    "ProcedureVRL checkpoint": PROCEDUREVRL_CKPT_PATH,
}

for name, path in required_paths.items():
    print(f"{name}: {path} -> exists={path.exists()}")
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required artifact: {name}: {path}"
        )

sequences = pd.read_csv(SEQUENCE_MANIFEST_PATH)
recordings = pd.read_csv(RECORDING_MANIFEST_PATH)
remote_sizes = pd.read_csv(
    REMOTE_SIZE_INVENTORY_PATH
)
notebook14_summary = json.loads(
    NOTEBOOK14_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)
class_metadata = pd.read_csv(
    SOURCE_CLASS_METADATA_PATH
)

required_sequence_columns = {
    "sequence_id",
    "sequence_filename",
    "split",
    "activity",
    "recording_name",
    "video_remote_path",
    "video_local_path",
    "clip_start_frame_30fps",
    "clip_end_frame_30fps_exclusive",
    "clip_start_seconds",
    "clip_end_seconds",
    "clip_duration_seconds",
    "ground_truth_path",
    "ground_truth_length_30fps",
}

missing_sequence_columns = (
    required_sequence_columns
    - set(sequences.columns)
)

if missing_sequence_columns:
    raise KeyError(
        "Sequence manifest is missing columns: "
        f"{sorted(missing_sequence_columns)}"
    )

if len(sequences) != 680:
    print(
        "WARNING: expected 680 sequences, found",
        len(sequences),
    )

if len(recordings) != 350:
    print(
        "WARNING: expected 350 recordings, found",
        len(recordings),
    )

if sequences["sequence_id"].duplicated().any():
    raise ValueError(
        "Duplicate sequence IDs in source manifest."
    )

if recordings["recording_name"].duplicated().any():
    raise ValueError(
        "Duplicate recording names in recording manifest."
    )

PINNED_REVISION = notebook14_summary[
    "pinned_revision"
]

print("Pinned Hugging Face revision:", PINNED_REVISION)
print("Sequences:", len(sequences))
print("Recordings:", len(recordings))
print(
    "Remote v1 size:",
    notebook14_summary["total_required_size_gib"],
    "GiB",
)
print(
    "Checkpoint size:",
    f"{PROCEDUREVRL_CKPT_PATH.stat().st_size / 1024**2:.2f} MiB",
)

display(sequences.head())

source mapping: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/mapping.txt -> exists=True
class metadata: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/class_metadata.csv -> exists=True
dense ground truth: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth -> exists=True
source splits: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/splits -> exists=True
sequence manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/procedurevrl_extraction_manifest_v1.csv -> exists=True
recording manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/recording_download_manifest_v1.csv -> exists=True
remote size inventory: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/remote_size_in

,sequence_id,sequence_filename,split,activity,is_shared,toy_id,toy_name,recording_name,selected_view,video_filename,video_remote_path,video_local_path,video_downloaded,clip_start_frame_30fps,clip_end_frame_30fps_exclusive,clip_start_seconds,clip_end_seconds,clip_duration_seconds,ground_truth_path,ground_truth_length_30fps,num_background_frames,output_feature_name,feature_extraction_status
0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,test,assembly,notshared,NaN,NaN,nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724/C10095_rgb.mp4,False,4457,8070,148.566667,269.000000,120.433333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.txt,3613,0,assembly_nusar-2021_action_both_9011-a01_9011_user_id_2021-02-01_153724.npy,video_not_downloaded
1,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt,train,assembly,-,b06b,-,nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253/C10095_rgb.mp4,False,2833,6959,94.433333,231.966667,137.533333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.txt,4126,0,assembly_nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253.npy,video_not_downloaded
2,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt,train,assembly,-,b08c,-,nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736/C10095_rgb.mp4,False,4777,11525,159.233333,384.166667,224.933333,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.txt,6748,0,assembly_nusar-2021_action_both_9011-b08c_9011_user_id_2021-02-01_154736.npy,video_not_downloaded
3,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.txt,test,assembly,notshared,NaN,NaN,nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620/C10095_rgb.mp4,False,4380,9978,146.000000,332.600000,186.600000,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/groundTruth/assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.txt,5598,0,assembly_nusar-2021_action_both_9011-c01c_9011_user_id_2021-02-01_155620.npy,video_not_downloaded
4,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,assembly_nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239.txt,val,assembly,notshared,c03f,roller,nusar-2021_action_both_9011-c03f_9011_user_id_2021-02-01_160239,v1,C10095_rgb.mp4,recordings/nusar-2021_action_both_9011

## 7. Verify gated file access

In [22]:
api = HfApi()

try:
    account = api.whoami(token=True)

    access_test = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        revision=PINNED_REVISION,
        filename="annotations/README.md",
        local_dir=ASSEMBLY_ROOT,
        token=True,
    )

    print("Hugging Face user:", account.get("name"))
    print("Assembly101 gated-file access: OK")
    print("Access test:", access_test)

except GatedRepoError as exc:
    raise RuntimeError(
        "The current Hugging Face account does not "
        "have Assembly101 access."
    ) from exc

except (
    RepositoryNotFoundError,
    HfHubHTTPError,
) as exc:
    raise RuntimeError(
        f"Assembly101 access check failed: {exc}"
    ) from exc

Hugging Face user: Bonart
Assembly101 gated-file access: OK
Access test: /content/drive/MyDrive/mmf_tas_lab_data/assembly101/annotations/README.md


## 8. Clone/reuse ProcedureVRL and apply the known Colab compatibility patches

### ProcedureVRL setup used in this V2 notebook

This cell deliberately treats `/content/ProcedureVRL` as a source checkout.
It verifies or reclones the repository, configures `PYTHONPATH`, applies the
known compatibility patches, and tests imports in both the notebook process
and a fresh subprocess.


In [23]:

def procedurevrl_checkout_is_complete(repo_path: Path) -> bool:
    required = [
        repo_path / ".git",
        repo_path / "lib",
        repo_path / "tools",
        repo_path / "configs",
        repo_path / "lib" / "datasets",
        repo_path / "lib" / "models",
    ]
    return all(path.exists() for path in required)


if PROCEDUREVRL_REPO.exists() and not procedurevrl_checkout_is_complete(
    PROCEDUREVRL_REPO
):
    print(
        "Existing ProcedureVRL directory is incomplete. "
        "Removing and cloning a clean checkout:"
    )
    print(PROCEDUREVRL_REPO)
    shutil.rmtree(PROCEDUREVRL_REPO)


if PROCEDUREVRL_REPO.exists():
    print("Reusing complete ProcedureVRL checkout:", PROCEDUREVRL_REPO)
else:
    print("Cloning ProcedureVRL:", PROCEDUREVRL_GIT)
    clone_result = subprocess.run(
        [
            "git",
            "clone",
            PROCEDUREVRL_GIT,
            str(PROCEDUREVRL_REPO),
        ],
        text=True,
        capture_output=True,
        check=False,
    )

    print(clone_result.stdout[-4000:])
    if clone_result.stderr:
        print(clone_result.stderr[-4000:])

    if clone_result.returncode != 0:
        raise RuntimeError(
            "ProcedureVRL clone failed with return code "
            f"{clone_result.returncode}."
        )


if not procedurevrl_checkout_is_complete(PROCEDUREVRL_REPO):
    raise RuntimeError(
        "ProcedureVRL checkout is still incomplete after setup."
    )


# ProcedureVRL is used as a source checkout. It is not installed as an
# editable Python package. The extraction scripts import modules from the
# repository root and from its lib/ directory.
PROCEDUREVRL_LIB = PROCEDUREVRL_REPO / "lib"

for path in [PROCEDUREVRL_REPO, PROCEDUREVRL_LIB]:
    value = str(path)
    if value not in sys.path:
        sys.path.insert(0, value)

pythonpath_parts = [
    str(PROCEDUREVRL_REPO),
    str(PROCEDUREVRL_LIB),
]

existing_pythonpath = os.environ.get("PYTHONPATH", "")
if existing_pythonpath:
    pythonpath_parts.append(existing_pythonpath)

os.environ["PYTHONPATH"] = ":".join(pythonpath_parts)

(PROCEDUREVRL_REPO / "exps").mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------------
# Idempotent compatibility patches used by the successful Breakfast run
# ------------------------------------------------------------------
feat_extract_path = PROCEDUREVRL_REPO / "tools" / "feat_extract.py"

if feat_extract_path.exists():
    original = feat_extract_path.read_text(
        encoding="utf-8",
        errors="replace",
    )
    patched = original.replace("np.int", "int")

    if patched != original:
        feat_extract_path.write_text(
            patched,
            encoding="utf-8",
        )
        print("Patched deprecated NumPy integer alias.")


video_builder_path = (
    PROCEDUREVRL_REPO
    / "lib"
    / "models"
    / "video_model_builder.py"
)
vit_path = (
    PROCEDUREVRL_REPO
    / "lib"
    / "models"
    / "vit.py"
)

if video_builder_path.exists():
    text = video_builder_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    old_import = (
        "from lib.models.vit "
        "import vit_base_patch16_224"
    )
    new_import = (
        "from lib.models.vit "
        "import vit_base_patch16_224_develop "
        "as vit_base_patch16_224"
    )

    if old_import in text and new_import not in text:
        video_builder_path.write_text(
            text.replace(old_import, new_import),
            encoding="utf-8",
        )
        print("Patched ProcedureVRL ViT builder import.")


if vit_path.exists():
    text = vit_path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    alias_marker = (
        "vit_base_patch16_224 = "
        "vit_base_patch16_224_develop"
    )

    if alias_marker not in text:
        alias_code = """

# Colab compatibility alias added by the Assembly101 notebook.
try:
    vit_base_patch16_224
except NameError:
    vit_base_patch16_224 = vit_base_patch16_224_develop
"""
        vit_path.write_text(
            text + alias_code,
            encoding="utf-8",
        )
        print("Added ProcedureVRL ViT compatibility alias.")




# Preserve fractional timestamps in the ProcedureVRL HowTo100M/COIN loader.
# The upstream loader converts duration/start/end to int, which unnecessarily
# truncates Assembly101 annotation times. Floats are accepted by all downstream
# timestamp and FFmpeg operations.
howto100m_path = (
    PROCEDUREVRL_REPO
    / "lib"
    / "datasets"
    / "howto100m.py"
)

if not howto100m_path.exists():
    raise FileNotFoundError(howto100m_path)

howto100m_text = howto100m_path.read_text(
    encoding="utf-8",
    errors="replace",
)

timestamp_replacements = {
    "self._durations.append(int(float(duration)))": (
        "self._durations.append(float(duration))"
    ),
    "self._start.append(int(float(start)))": (
        "self._start.append(float(start))"
    ),
    "self._end.append(int(float(end)))": (
        "self._end.append(float(end))"
    ),
}

howto100m_patched = howto100m_text

for old, new in timestamp_replacements.items():
    howto100m_patched = howto100m_patched.replace(old, new)

if howto100m_patched != howto100m_text:
    howto100m_path.write_text(
        howto100m_patched,
        encoding="utf-8",
    )
    print(
        "Patched ProcedureVRL loader to preserve "
        "fractional duration/start/end timestamps."
    )

howto100m_check = howto100m_path.read_text(
    encoding="utf-8",
    errors="replace",
)

for forbidden in [
    "self._durations.append(int(float(duration)))",
    "self._start.append(int(float(start)))",
    "self._end.append(int(float(end)))",
]:
    if forbidden in howto100m_check:
        raise RuntimeError(
            "ProcedureVRL timestamp patch was not applied: "
            + forbidden
        )


git_commit_result = subprocess.run(
    [
        "git",
        "-C",
        str(PROCEDUREVRL_REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
    capture_output=True,
    check=False,
)

if git_commit_result.returncode != 0:
    raise RuntimeError(
        "Could not read the ProcedureVRL Git revision:\n"
        + git_commit_result.stderr[-2000:]
    )

git_commit = git_commit_result.stdout.strip()

print("ProcedureVRL Git commit:", git_commit)
print("ProcedureVRL root:", PROCEDUREVRL_REPO)
print("ProcedureVRL lib:", PROCEDUREVRL_LIB)
print("PYTHONPATH:", os.environ["PYTHONPATH"])


# Validate imports in the current notebook process.
for module_name in [
    "clip",
    "decord",
    "pytorchvideo",
    "lib",
    "lib.datasets",
    "lib.models",
]:
    spec = importlib.util.find_spec(module_name)
    print(module_name, "->", spec is not None)

    if spec is None:
        raise ImportError(
            f"Required module is unavailable: {module_name}"
        )


# Validate imports in a fresh subprocess as well. This catches environment
# errors before the much more expensive feature-extraction stage.
import_check_code = """
import sys
import clip
import decord
import pytorchvideo
import lib
import lib.datasets
import lib.models
print("fresh subprocess imports: OK")
print("python:", sys.executable)
"""

import_check = subprocess.run(
    [sys.executable, "-c", import_check_code],
    cwd=str(PROCEDUREVRL_REPO),
    env={
        **os.environ,
        "PYTHONPATH": os.environ["PYTHONPATH"],
    },
    text=True,
    capture_output=True,
    check=False,
)

print(import_check.stdout)
if import_check.stderr:
    print(import_check.stderr[-4000:])

if import_check.returncode != 0:
    raise RuntimeError(
        "ProcedureVRL fresh-subprocess import check failed "
        f"with return code {import_check.returncode}."
    )

print("ProcedureVRL source-checkout setup: OK")


Reusing complete ProcedureVRL checkout: /content/ProcedureVRL
ProcedureVRL Git commit: fcbf833bf469d42f0d657be5ff7ec092d68e65aa
ProcedureVRL root: /content/ProcedureVRL
ProcedureVRL lib: /content/ProcedureVRL/lib
PYTHONPATH: /content/ProcedureVRL:/content/ProcedureVRL/lib:/content/ProcedureVRL:/content/ProcedureVRL/lib:/env/python
clip -> True
decord -> True
pytorchvideo -> True
lib -> True
lib.datasets -> True
lib.models -> True
fresh subprocess imports: OK
python: /usr/bin/python3

ProcedureVRL source-checkout setup: OK


## 9. Copy mapping metadata into both self-contained MS-TCN dataset roots

In [24]:
for dataset_root in [
    PROC_DATASET_ROOT,
    CLIP_DATASET_ROOT,
]:
    shutil.copy2(
        SOURCE_MAPPING_PATH,
        dataset_root / "mapping.txt",
    )
    shutil.copy2(
        SOURCE_CLASS_METADATA_PATH,
        dataset_root / "class_metadata.csv",
    )

print("Copied mapping and class metadata.")

Copied mapping and class metadata.


## 10. Feature/ground-truth validation helpers

In [25]:
def read_nonempty_lines(path):
    path = Path(path)
    return [
        line.strip()
        for line in path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip()
    ]


def load_mapping_labels(path):
    labels = set()

    for line in read_nonempty_lines(path):
        _, label = line.split(maxsplit=1)
        labels.add(label)

    return labels


MAPPING_LABELS = load_mapping_labels(
    SOURCE_MAPPING_PATH
)


def inspect_feature(path, expected_dim):
    path = Path(path)

    result = {
        "exists": path.exists(),
        "shape": None,
        "dim": None,
        "length": None,
        "finite": False,
        "valid": False,
        "error": "",
    }

    if not path.exists():
        return result

    try:
        array = np.load(
            path,
            mmap_mode="r",
        )

        result["shape"] = list(array.shape)

        if array.ndim == 2:
            result["dim"] = int(array.shape[0])
            result["length"] = int(array.shape[1])

        finite = bool(np.isfinite(array).all())
        result["finite"] = finite

        result["valid"] = bool(
            array.ndim == 2
            and array.shape[0] == expected_dim
            and array.shape[1] == NUM_TEMPORAL_STEPS
            and finite
        )

    except Exception as exc:
        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    return result


def inspect_coarse_gt(path):
    path = Path(path)

    result = {
        "exists": path.exists(),
        "length": 0,
        "unknown_labels": [],
        "valid": False,
        "error": "",
    }

    if not path.exists():
        return result

    try:
        labels = read_nonempty_lines(path)
        unknown = sorted(
            set(labels) - MAPPING_LABELS
        )

        result["length"] = len(labels)
        result["unknown_labels"] = unknown
        result["valid"] = bool(
            len(labels) == NUM_TEMPORAL_STEPS
            and len(unknown) == 0
        )

    except Exception as exc:
        result["error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    return result


def rebuild_sequence_status():
    rows = []

    for row in sequences.itertuples(index=False):
        proc_path = (
            PROC_FEATURE_DIR
            / f"{row.sequence_id}.npy"
        )
        clip_path = (
            CLIP_FEATURE_DIR
            / f"{row.sequence_id}.npy"
        )
        proc_gt_path = (
            PROC_GT_DIR
            / f"{row.sequence_id}.txt"
        )
        clip_gt_path = (
            CLIP_GT_DIR
            / f"{row.sequence_id}.txt"
        )

        proc = inspect_feature(
            proc_path,
            EXPECTED_PROCEDUREVRL_DIM,
        )
        clip_result = inspect_feature(
            clip_path,
            EXPECTED_CLIP_DIM,
        )
        proc_gt = inspect_coarse_gt(
            proc_gt_path
        )
        clip_gt = inspect_coarse_gt(
            clip_gt_path
        )

        gt_equal = False

        if proc_gt["valid"] and clip_gt["valid"]:
            gt_equal = (
                read_nonempty_lines(proc_gt_path)
                == read_nonempty_lines(clip_gt_path)
            )

        complete = bool(
            proc["valid"]
            and clip_result["valid"]
            and proc_gt["valid"]
            and clip_gt["valid"]
            and gt_equal
        )

        rows.append({
            "sequence_id": row.sequence_id,
            "recording_name": row.recording_name,
            "split": row.split,
            "activity": row.activity,
            "procedurevrl_feature_path": str(
                proc_path
            ),
            "procedurevrl_valid": proc["valid"],
            "procedurevrl_shape": (
                json.dumps(proc["shape"])
                if proc["shape"] is not None
                else ""
            ),
            "clip_feature_path": str(
                clip_path
            ),
            "clip_valid": clip_result["valid"],
            "clip_shape": (
                json.dumps(clip_result["shape"])
                if clip_result["shape"] is not None
                else ""
            ),
            "procedurevrl_gt_path": str(
                proc_gt_path
            ),
            "procedurevrl_gt_valid": proc_gt["valid"],
            "clip_gt_path": str(
                clip_gt_path
            ),
            "clip_gt_valid": clip_gt["valid"],
            "gt_equal_between_representations": (
                gt_equal
            ),
            "complete_all": complete,
            "last_checked_utc": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
        })

    frame = pd.DataFrame(rows)
    frame.to_csv(
        SEQUENCE_STATUS_PATH,
        index=False,
    )
    return frame


def rebuild_recording_status(sequence_status):
    status = (
        sequence_status.groupby(
            "recording_name",
            as_index=False,
        )
        .agg(
            num_sequences=(
                "sequence_id",
                "size",
            ),
            num_sequences_complete=(
                "complete_all",
                "sum",
            ),
            procedurevrl_sequences_complete=(
                "procedurevrl_valid",
                "sum",
            ),
            clip_sequences_complete=(
                "clip_valid",
                "sum",
            ),
        )
    )

    status["complete_all"] = (
        status["num_sequences_complete"]
        == status["num_sequences"]
    )

    recording_paths = (
        sequences[
            [
                "recording_name",
                "video_remote_path",
                "video_local_path",
            ]
        ]
        .drop_duplicates(
            subset=["recording_name"]
        )
    )

    status = status.merge(
        recording_paths,
        on="recording_name",
        how="left",
        validate="one_to_one",
    )

    size_columns = remote_sizes[
        [
            "recording_name",
            "remote_size_bytes",
            "remote_size_gib",
        ]
    ].drop_duplicates(
        subset=["recording_name"]
    )

    status = status.merge(
        size_columns,
        on="recording_name",
        how="left",
        validate="one_to_one",
    )

    status["raw_video_exists"] = (
        status["video_local_path"]
        .map(lambda value: Path(value).exists())
    )

    status["raw_local_size_bytes"] = (
        status["video_local_path"]
        .map(
            lambda value: (
                Path(value).stat().st_size
                if Path(value).exists()
                else 0
            )
        )
    )

    status["raw_size_matches"] = (
        status["raw_video_exists"]
        & (
            status["raw_local_size_bytes"]
            == status["remote_size_bytes"]
        )
    )

    status["last_checked_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )

    status.to_csv(
        RECORDING_STATUS_PATH,
        index=False,
    )
    return status


sequence_status = rebuild_sequence_status()
recording_status = rebuild_recording_status(
    sequence_status
)

print(
    "Complete sequences:",
    int(sequence_status["complete_all"].sum()),
    "/",
    len(sequence_status),
)
print(
    "Complete recordings:",
    int(recording_status["complete_all"].sum()),
    "/",
    len(recording_status),
)
print(
    "Raw recordings currently available:",
    int(recording_status["raw_size_matches"].sum()),
)

display(
    recording_status.sort_values(
        [
            "complete_all",
            "raw_size_matches",
            "remote_size_bytes",
        ],
        ascending=[True, False, True],
    ).head(20)
)

Complete sequences: 680 / 680
Complete recordings: 350 / 350
Raw recordings currently available: 0


,recording_name,num_sequences,num_sequences_complete,procedurevrl_sequences_complete,clip_sequences_complete,complete_all,video_remote_path,video_local_path,remote_size_bytes,remote_size_gib,raw_video_exists,raw_local_size_bytes,raw_size_matches,last_checked_utc
334,nusar-2021_action_both_9084-b04c_9084_user_id_2021-02-25_143221,1,1,1,1,True,recordings/nusar-2021_action_both_9084-b04c_9084_user_id_2021-02-25_143221/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9084-b04c_9084_user_id_2021-02-25_143221/C10095_rgb.mp4,187626993,0.174741,False,0,False,2026-07-24T07:50:03.517009+00:00
61,nusar-2021_action_both_9023-b05d_9023_user_id_2021-02-23_135325,1,1,1,1,True,recordings/nusar-2021_action_both_9023-b05d_9023_user_id_2021-02-23_135325/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-b05d_9023_user_id_2021-02-23_135325/C10095_rgb.mp4,239689856,0.223229,False,0,False,2026-07-24T07:50:03.517009+00:00
63,nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136,2,2,2,2,True,recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9023-c04b_9023_user_id_2021-02-23_144136/C10095_rgb.mp4,275821971,0.256879,False,0,False,2026-07-24T07:50:03.517009+00:00
202,nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244,2,2,2,2,True,recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9052-c03d_9052_user_id_2021-02-25_170244/C10095_rgb.mp4,282760168,0.263341,False,0,False,2026-07-24T07:50:03.517009+00:00
342,nusar-2021_action_both_9085-c03d_9085_user_id_2021-02-22_174243,2,2,2,2,True,recordings/nusar-2021_action_both_9085-c03d_9085_user_id_2021-02-22_174243/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9085-c03d_9085_user_id_2021-02-22_174243/C10095_rgb.mp4,303183792,0.282362,False,0,False,2026-07-24T07:50:03.517009+00:00
50,nusar-2021_action_both_9022-b03b_9022_user_id_2021-02-23_105258,2,2,2,2,True,recordings/nusar-2021_action_both_9022-b03b_9022_user_id_2021-02-23_105258/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9022-b03b_9022_user_id_2021-02-23_105258/C10095_rgb.mp4,314333874,0.292746,False,0,False,2026-07-24T07:50:03.517009+00:00
224,nusar-2021_action_both_9055-b08b_9055_user_id_2021-02-24_104106,2,2,2,2,True,recordings/nusar-2021_action_both_9055-b08b_9055_user_id_2021-02-24_104106/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9055-b08b_9055_user_id_2021-02-24_104106/C10095_rgb.mp4,326270709,0.303863,False,0,False,2026-07-24T07:50:03.517009+00:00
134,nusar-2021_action_both_9034-c13f_9034_user_id_2021-02-23_180813,2,2,2,2,True,recordings/nusar-2021_action_both_9034-c13f_9034_user_id_2021-02-23_180813/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9034-c13f_9034_user_id_2021-02-23_180813/C10095_rgb.mp4,369538360,0.344159,False,0,False,2026-07-24T07:50:03.517009+00:00
257,nusar-2021_action_both_9064-a20_9064_user_id_2021-02-22_161628,2,2,2,2,True,recordings/nusar-2021_action_both_9064-a20_9064_user_id_2021-02-22_161628/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9064-a20_9064_user_id_2021-02-22_161628/C10095_rgb.mp4,370364462,0.344929,False,0,False,2026-07-24T07:50:03.517009+00:00
131,nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,2,2,2,2,True,recordings/nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828/C10095_rgb.mp4,/content/drive/MyDrive/mmf_tas_lab_data/assembly101/recordings/nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828/C10095_rgb.mp4,371231185,0.345736,


## 11. Prepare models once for the automatic session

The ProcedureVRL extraction script is written once. CLIP and the 202 action
text embeddings are loaded once and reused by every automatic batch.


In [26]:
hidden_script_path = (
    PROCEDUREVRL_REPO
    / "tools"
    / "feat_extract_hidden_streaming.py"
)

hidden_script = r"""
#!/usr/bin/env python3
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

from lib.datasets import loader
from lib.models import build_model
import lib.utils.checkpoint as cu
import lib.utils.logging as logging
from lib.utils.misc import launch_job
from lib.utils.parser import parse_args, load_config

_CAPTURE = {}


def _recursive_to_cuda(value):
    if isinstance(value, torch.Tensor):
        if torch.cuda.is_available():
            return value.cuda(non_blocking=True)
        return value

    if isinstance(value, list):
        return [
            _recursive_to_cuda(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            _recursive_to_cuda(item)
            for item in value
        )

    if isinstance(value, dict):
        return {
            key: _recursive_to_cuda(item)
            for key, item in value.items()
        }

    return value


def _as_numpy_index(video_idx):
    if isinstance(video_idx, torch.Tensor):
        return (
            video_idx.detach()
            .cpu()
            .numpy()
            .astype(np.int64)
        )

    return np.asarray(
        video_idx,
        dtype=np.int64,
    )


def _choose_hidden_projection(model):
    modules = dict(model.named_modules())

    # Exact architecture observed with the validated checkpoint:
    # model.head = Linear(in_features=768, out_features=512, bias=True)
    exact_names = [
        "model.head",
        "head",
        "video_encoder.head",
        "video_encoder.model.head",
    ]

    for name in exact_names:
        module = modules.get(name)

        if (
            isinstance(module, nn.Linear)
            and int(module.out_features) == 512
        ):
            return {
                "name": name,
                "type": type(module).__name__,
                "in_features": int(module.in_features),
                "out_features": int(module.out_features),
                "selection_rule": "exact_known_projection_name",
            }

    # Robust fallback: prefer a 512-dimensional non-text Linear module whose
    # name contains "head". This still avoids CLIP text-transformer projections.
    candidates = []

    for name, module in model.named_modules():
        if not isinstance(module, nn.Linear):
            continue

        if int(module.out_features) != 512:
            continue

        lower_name = name.lower()

        if "text_model" in lower_name:
            continue

        score = 100

        if lower_name == "model.head":
            score = 0
        elif lower_name.endswith(".head"):
            score = 5
        elif "head" in lower_name:
            score = 10
        elif "projection" in lower_name:
            score = 20
        elif "proj" in lower_name:
            score = 30

        candidates.append({
            "name": name,
            "type": type(module).__name__,
            "in_features": int(module.in_features),
            "out_features": int(module.out_features),
            "score": score,
            "selection_rule": "ranked_512d_non_text_linear",
        })

    candidates = sorted(
        candidates,
        key=lambda item: (
            item["score"],
            len(item["name"]),
            item["name"],
        ),
    )

    if not candidates:
        diagnostic = []

        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                diagnostic.append({
                    "name": name,
                    "in_features": int(module.in_features),
                    "out_features": int(module.out_features),
                })

        raise RuntimeError(
            "No suitable 512-dimensional ProcedureVRL video projection "
            "was found. Linear-module diagnostics: "
            + json.dumps(diagnostic[:120], indent=2)
        )

    selected = dict(candidates[0])
    selected.pop("score", None)
    return selected


def _register_projection_output_hook(model, target_name):
    target_module = dict(model.named_modules())[target_name]

    def forward_hook(module, inputs, output):
        value = output

        # Some modules may return tuples. The ProcedureVRL projection head
        # normally returns a tensor directly.
        if isinstance(value, (tuple, list)):
            tensor_values = [
                item
                for item in value
                if isinstance(item, torch.Tensor)
            ]
            if not tensor_values:
                return
            value = tensor_values[0]

        if isinstance(value, torch.Tensor):
            _CAPTURE["hidden"] = value.detach()
            _CAPTURE["raw_shape"] = list(value.shape)
            _CAPTURE["target_name"] = target_name
            _CAPTURE["capture_kind"] = "module_output"

    return target_module.register_forward_hook(
        forward_hook
    )

def _normalize_hidden_tensor(value):
    if not isinstance(value, torch.Tensor):
        raise TypeError(
            "Captured hidden value is not a tensor: "
            f"{type(value)}"
        )

    if value.ndim == 2:
        normalized = value
    elif value.ndim == 3:
        normalized = value.mean(dim=1)
    else:
        normalized = value.reshape(
            value.shape[0],
            -1,
        )

    return (
        normalized.detach()
        .float()
        .cpu()
        .numpy()
    )


@torch.no_grad()
def perform_hidden_extraction(
    test_loader,
    model,
    cfg,
):
    model.eval()

    n_clip = int(
        cfg.TEST.NUM_ENSEMBLE_VIEWS
        * cfg.TEST.NUM_SPATIAL_CROPS
    )
    n_samples = len(test_loader.dataset)

    if n_samples % n_clip != 0:
        raise RuntimeError(
            f"Dataset size {n_samples} is not "
            f"divisible by n_clip {n_clip}."
        )

    n_video = n_samples // n_clip

    print(
        f"n_samples={n_samples}; "
        f"n_video={n_video}; "
        f"n_clip={n_clip}"
    )

    target = _choose_hidden_projection(model)

    print(
        "Selected ProcedureVRL hidden projection:",
        json.dumps(target, indent=2),
    )

    handle = _register_projection_output_hook(
        model,
        target["name"],
    )

    hidden_store = None
    filled = np.zeros(
        (n_video, n_clip),
        dtype=np.int32,
    )
    raw_hidden_shapes = []

    for cur_iter, (
        inputs,
        labels,
        video_idx,
        meta,
    ) in enumerate(tqdm(test_loader)):
        inputs = _recursive_to_cuda(inputs)
        indices = _as_numpy_index(video_idx)

        _CAPTURE.clear()
        _ = model(inputs)

        if "hidden" not in _CAPTURE:
            raise RuntimeError(
                "No 512-dimensional projection output was captured at "
                f"iteration {cur_iter}. Target={target}"
            )

        hidden = _normalize_hidden_tensor(
            _CAPTURE["hidden"]
        )
        raw_hidden_shapes.append(
            _CAPTURE.get("raw_shape")
        )

        if hidden.ndim != 2 or hidden.shape[1] != 512:
            raise RuntimeError(
                "Captured ProcedureVRL projection has unexpected shape "
                f"{hidden.shape}; expected [batch, 512]. "
                f"Target={target}"
            )

        if hidden_store is None:
            hidden_dim = int(hidden.shape[1])
            hidden_store = np.zeros(
                (
                    n_video,
                    n_clip,
                    hidden_dim,
                ),
                dtype=np.float32,
            )
            print(
                "hidden_store shape:",
                hidden_store.shape,
            )

        if hidden.shape[0] != len(indices):
            raise RuntimeError(
                f"Batch mismatch: hidden "
                f"{hidden.shape}, indices "
                f"{indices.shape}"
            )

        video_indices = indices // n_clip
        clip_indices = indices % n_clip

        if video_indices.max() >= n_video:
            raise RuntimeError(
                "video_idx mapping failed: "
                f"max={indices.max()}, "
                f"n_video={n_video}, "
                f"n_clip={n_clip}"
            )

        for batch_index in range(
            hidden.shape[0]
        ):
            video_index = int(
                video_indices[batch_index]
            )
            clip_index = int(
                clip_indices[batch_index]
            )

            hidden_store[
                video_index,
                clip_index,
                :,
            ] = hidden[batch_index]

            filled[
                video_index,
                clip_index,
            ] += 1

    handle.remove()

    if hidden_store is None:
        raise RuntimeError(
            "No hidden embeddings were extracted."
        )

    missing = int((filled == 0).sum())
    duplicates = int((filled > 1).sum())

    print("filled missing:", missing)
    print("filled duplicates:", duplicates)

    if missing != 0:
        raise RuntimeError(
            "Some video/clip positions were not "
            f"filled: {missing}"
        )

    if duplicates != 0:
        raise RuntimeError(
            "Duplicate video/clip positions were "
            f"filled: {duplicates}"
        )

    return hidden_store, {
        "hook_target": target,
        "projection_target": target,
        "n_video": int(n_video),
        "n_clip": int(n_clip),
        "hidden_dim": int(
            hidden_store.shape[2]
        ),
        "raw_hidden_shapes_seen": (
            raw_hidden_shapes[:20]
        ),
        "filled_missing": missing,
        "filled_duplicates": duplicates,
    }


def test(cfg):
    logging.setup_logging(cfg.OUTPUT_DIR)

    print("Building model.")
    model = build_model(cfg)

    if torch.cuda.is_available():
        model = model.cuda()

    print("Loading checkpoint.")
    cu.load_test_checkpoint(cfg, model)

    print("Constructing test loader.")
    test_loader = loader.construct_loader(
        cfg,
        "test",
    )

    hidden_store, summary = (
        perform_hidden_extraction(
            test_loader,
            model,
            cfg,
        )
    )

    save_path = Path(
        cfg.TEST.SAVE_PREDICT_PATH
    )
    save_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.save(save_path, hidden_store)

    summary_path = save_path.with_suffix(
        ".summary.json"
    )
    summary_path.write_text(
        json.dumps(summary, indent=2)
    )

    print(
        "Saved hidden embeddings:",
        save_path,
    )
    print("Saved summary:", summary_path)
    print(json.dumps(summary, indent=2))


def main():
    args = parse_args()
    cfg = load_config(args)

    launch_job(
        cfg=cfg,
        init_method=args.init_method,
        func=test,
    )


if __name__ == "__main__":
    main()
"""

hidden_script_path.write_text(
    hidden_script,
    encoding="utf-8",
)

print("Wrote:", hidden_script_path)
print(
    "Script size:",
    hidden_script_path.stat().st_size,
)

Wrote: /content/ProcedureVRL/tools/feat_extract_hidden_streaming.py
Script size: 10930


In [27]:
import clip

CLIP_DEVICE = torch.device("cuda")

clip_model, clip_preprocess = clip.load(
    CLIP_MODEL_NAME,
    device=CLIP_DEVICE,
    jit=False,
)

clip_model.eval()

clip_parameter_count = sum(
    parameter.numel()
    for parameter in clip_model.parameters()
)

print("CLIP model:", CLIP_MODEL_NAME)
print("CLIP device:", CLIP_DEVICE)
print("Parameters:", clip_parameter_count)
print(
    "Visual output dimension:",
    getattr(clip_model.visual, "output_dim", None),
)

CLIP model: ViT-B/16
CLIP device: cuda
Parameters: 149620737
Visual output dimension: 512


In [28]:
action_rows = class_metadata[
    ~class_metadata["is_background"].astype(bool)
].copy()

action_rows = action_rows.sort_values(
    "model_class_id"
).reset_index(drop=True)

action_labels = action_rows[
    "action_cls"
].astype(str).tolist()

prompts = [
    TEXT_PROMPT_TEMPLATE.format(label)
    for label in action_labels
]

with torch.no_grad():
    text_tokens = clip.tokenize(
        prompts,
        truncate=True,
    ).to(CLIP_DEVICE)

    text_embeddings = clip_model.encode_text(
        text_tokens
    ).float()

    text_embeddings = (
        text_embeddings
        / text_embeddings.norm(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-12)
    )

text_embeddings_np = (
    text_embeddings.cpu().numpy()
    .astype(np.float32)
)

if text_embeddings_np.shape != (
    len(action_labels),
    EXPECTED_CLIP_DIM,
):
    raise RuntimeError(
        "Unexpected CLIP text embedding shape: "
        f"{text_embeddings_np.shape}"
    )

clip_text_embedding_path = (
    CLIP_TEXT_DIR
    / "assembly101_coarse_clip_vitb16_text_embeddings.npy"
)
clip_text_metadata_path = (
    CLIP_TEXT_DIR
    / "assembly101_coarse_clip_vitb16_text_metadata.csv"
)

np.save(
    clip_text_embedding_path,
    text_embeddings_np,
)

text_metadata = action_rows[
    [
        "model_class_id",
        "official_action_id",
        "action_cls",
        "verb_cls",
        "noun_cls",
    ]
].copy()
text_metadata["prompt"] = prompts
text_metadata["clip_model"] = CLIP_MODEL_NAME
text_metadata.to_csv(
    clip_text_metadata_path,
    index=False,
)

print(
    "CLIP text embeddings:",
    text_embeddings_np.shape,
)
print("Saved:", clip_text_embedding_path)
print("Saved:", clip_text_metadata_path)

display(text_metadata.head(20))

CLIP text embeddings: (202, 512)
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16/text_embeddings/assembly101_coarse_clip_vitb16_text_embeddings.npy
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16/text_embeddings/assembly101_coarse_clip_vitb16_text_metadata.csv


,model_class_id,official_action_id,action_cls,verb_cls,noun_cls,prompt,clip_model
0,0,0,inspect toy,inspect,toy,a video of a person performing the action: inspect toy,ViT-B/16
1,1,1,attach cabin,attach,cabin,a video of a person performing the action: attach cabin,ViT-B/16
2,2,2,detach cabin,detach,cabin,a video of a person performing the action: detach cabin,ViT-B/16
3,3,3,detach wheel,detach,wheel,a video of a person performing the action: detach wheel,ViT-B/16
4,4,4,attach wheel,attach,wheel,a video of a person performing the action: attach wheel,ViT-B/16
5,5,5,screw chassis,screw,chassis,a video of a person performing the action: screw chassis,ViT-B/16
6,6,6,demonstrate functionality,demonstrate,functionality,a video of a person performing the action: demonstrate functionality,ViT-B/16
7,7,7,unscrew chassis,unscrew,chassis,a video of a person performing the action: unscrew chassis,ViT-B/16
8,8,8,attach interior,attach,interior,a video of a person performing the action: attach interior,ViT-B/16
9,9,9,detach roof,detach,roof,a video of a person performing the action: detach roof,ViT-B/16



## 12. Define one validated streaming batch

The function below reuses the V4 single-batch implementation:

- select unfinished recordings;
- download the current batch;
- run FFprobe and interval validation;
- extract ProcedureVRL `[512, 16]`;
- extract CLIP ViT-B/16 `[512, 16]`;
- align 16 coarse labels;
- validate every output;
- update persistent progress;
- delete raw videos only after complete validation.


In [29]:
def process_one_stream_batch():
    """Process exactly one persistent Assembly101 batch."""
    global sequence_status
    global recording_status

    # ===== Validated V4 code cell 23 =====
    incomplete_recordings = recording_status[
        ~recording_status["complete_all"]
    ].copy()

    if incomplete_recordings.empty:
        selected_recordings = (
            recording_status.iloc[0:0].copy()
        )
        print(
            "All recordings already have both feature "
            "representations."
        )

    elif RUN_MODE == "smoke":
        # Prefer the videos already downloaded by notebook 14.
        local_candidates = incomplete_recordings[
            incomplete_recordings["raw_size_matches"]
        ].sort_values(
            [
                "num_sequences",
                "remote_size_bytes",
                "recording_name",
            ],
            ascending=[False, True, True],
        )

        other_candidates = incomplete_recordings[
            ~incomplete_recordings["raw_size_matches"]
        ].sort_values(
            [
                "num_sequences",
                "remote_size_bytes",
                "recording_name",
            ],
            ascending=[False, True, True],
        )

        selected_recordings = (
            pd.concat(
                [
                    local_candidates,
                    other_candidates,
                ],
                ignore_index=True,
            )
            .drop_duplicates(
                subset=["recording_name"]
            )
            .head(SMOKE_RECORDINGS)
        )

    else:
        # Smallest incomplete recordings first keeps the batch footprint low.
        selected_recordings = (
            incomplete_recordings.sort_values(
                [
                    "remote_size_bytes",
                    "recording_name",
                ]
            )
            .head(STREAM_BATCH_RECORDINGS)
        )

    selected_recordings = (
        selected_recordings.copy()
    )

    selected_names = selected_recordings[
        "recording_name"
    ].tolist()

    selected_sequences = (
        sequences[
            sequences["recording_name"].isin(
                set(selected_names)
            )
        ]
        .sort_values(
            [
                "recording_name",
                "clip_start_seconds",
                "sequence_id",
            ]
        )
        .reset_index(drop=True)
    )

    selected_sequences["run_index"] = np.arange(
        len(selected_sequences)
    )

    batch_key_source = "|".join(selected_names)
    batch_hash = hashlib.sha1(
        batch_key_source.encode("utf-8")
    ).hexdigest()[:12]

    BATCH_NAME = (
        f"{RUN_MODE}_"
        f"{len(selected_recordings)}rec_"
        f"{batch_hash}"
    )

    BATCH_ROOT = RUNS_ROOT / BATCH_NAME
    CSV_DIR = BATCH_ROOT / "data_csv"
    RAW_OUTPUT_DIR = BATCH_ROOT / "raw_outputs"
    PROC_OUTPUT_DIR = (
        BATCH_ROOT / "procedurevrl_output"
    )

    for path in [
        BATCH_ROOT,
        CSV_DIR,
        RAW_OUTPUT_DIR,
        PROC_OUTPUT_DIR,
    ]:
        path.mkdir(parents=True, exist_ok=True)

    BATCH_RECORDING_MANIFEST_PATH = (
        BATCH_ROOT / "batch_recordings.csv"
    )
    BATCH_SEQUENCE_MANIFEST_PATH = (
        BATCH_ROOT / "batch_sequences.csv"
    )
    BATCH_HIDDEN_NPY = (
        RAW_OUTPUT_DIR
        / "procedurevrl_hidden_embeddings.npy"
    )
    BATCH_LOG_PATH = (
        BATCH_ROOT
        / "procedurevrl_hidden_extract.log"
    )
    BATCH_ALIGNMENT_PATH = (
        BATCH_ROOT
        / "batch_temporal_alignment.csv"
    )
    BATCH_SUMMARY_PATH = (
        BATCH_ROOT
        / "batch_summary.json"
    )

    selected_recordings.to_csv(
        BATCH_RECORDING_MANIFEST_PATH,
        index=False,
    )
    selected_sequences.to_csv(
        BATCH_SEQUENCE_MANIFEST_PATH,
        index=False,
    )

    new_download_bytes = int(
        selected_recordings.loc[
            ~selected_recordings["raw_size_matches"],
            "remote_size_bytes",
        ].sum()
    )

    print("Batch:", BATCH_NAME)
    print("Selected recordings:", len(selected_recordings))
    print("Selected sequences:", len(selected_sequences))
    print(
        "New download required:",
        f"{new_download_bytes / 1024**3:.3f} GiB",
    )
    print("DELETE_RAW_AFTER_SUCCESS:", DELETE_RAW_AFTER_SUCCESS)

    display(
        selected_recordings[
            [
                "recording_name",
                "num_sequences",
                "raw_size_matches",
                "remote_size_gib",
                "complete_all",
            ]
        ]
    )

    display(
        selected_sequences[
            [
                "run_index",
                "sequence_id",
                "split",
                "activity",
                "recording_name",
                "clip_start_seconds",
                "clip_end_seconds",
            ]
        ]
    )

    # ===== Validated V4 code cell 25 =====
    def download_one_recording(row):
        target_path = Path(row.video_local_path)
        expected_size = int(row.remote_size_bytes)

        if (
            target_path.exists()
            and target_path.stat().st_size
            == expected_size
        ):
            return {
                "recording_name": row.recording_name,
                "status": "already_complete",
                "path": str(target_path),
                "size_bytes": expected_size,
                "error": "",
            }

        if target_path.exists():
            print(
                "Removing incomplete target:",
                target_path,
            )
            target_path.unlink()

        last_error = None

        for attempt in range(
            1,
            MAX_DOWNLOAD_ATTEMPTS + 1,
        ):
            try:
                print(
                    f"\nDownloading {row.recording_name} "
                    f"attempt {attempt}/"
                    f"{MAX_DOWNLOAD_ATTEMPTS}"
                )
                print(
                    "Expected:",
                    f"{expected_size / 1024**3:.3f} GiB",
                )

                returned_path = Path(
                    hf_hub_download(
                        repo_id=REPO_ID,
                        repo_type=REPO_TYPE,
                        revision=PINNED_REVISION,
                        filename=row.video_remote_path,
                        local_dir=ASSEMBLY_ROOT,
                        token=True,
                    )
                )

                if not target_path.exists():
                    raise FileNotFoundError(
                        f"Target missing after download: "
                        f"{target_path}; returned "
                        f"{returned_path}"
                    )

                actual_size = target_path.stat().st_size

                if actual_size != expected_size:
                    raise IOError(
                        f"Size mismatch: {actual_size} "
                        f"!= {expected_size}"
                    )

                return {
                    "recording_name": row.recording_name,
                    "status": "downloaded",
                    "path": str(target_path),
                    "size_bytes": actual_size,
                    "error": "",
                }

            except Exception as exc:
                last_error = (
                    f"{type(exc).__name__}: {exc}"
                )
                print("Attempt failed:", last_error)

                if attempt < MAX_DOWNLOAD_ATTEMPTS:
                    time.sleep(
                        RETRY_SLEEP_SECONDS
                    )

        return {
            "recording_name": row.recording_name,
            "status": "failed",
            "path": str(target_path),
            "size_bytes": (
                target_path.stat().st_size
                if target_path.exists()
                else 0
            ),
            "error": last_error or "unknown",
        }


    download_rows = []

    for row in tqdm(
        selected_recordings.itertuples(index=False),
        total=len(selected_recordings),
        desc="Raw recordings",
    ):
        download_rows.append(
            download_one_recording(row)
        )

    batch_download_manifest = pd.DataFrame(
        download_rows
    )

    batch_download_manifest_path = (
        BATCH_ROOT / "batch_download_manifest.csv"
    )
    batch_download_manifest.to_csv(
        batch_download_manifest_path,
        index=False,
    )

    display(batch_download_manifest)

    if not batch_download_manifest.empty:
        failed_downloads = batch_download_manifest[
            batch_download_manifest["status"]
            == "failed"
        ]

        if not failed_downloads.empty:
            raise RuntimeError(
                "At least one batch recording failed "
                "to download."
            )

    print("Saved:", batch_download_manifest_path)

    # ===== Validated V4 code cell 27 =====
    def parse_fraction(value):
        if value in (None, "", "0/0"):
            return np.nan

        try:
            return float(Fraction(str(value)))
        except Exception:
            return np.nan


    def ffprobe_video(path):
        command = [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "v:0",
            "-show_entries",
            (
                "stream=codec_name,width,height,"
                "r_frame_rate,avg_frame_rate,"
                "nb_frames,duration:"
                "format=duration,size"
            ),
            "-of",
            "json",
            str(path),
        ]

        completed = subprocess.run(
            command,
            text=True,
            capture_output=True,
            check=False,
        )

        payload = (
            json.loads(completed.stdout)
            if completed.stdout.strip()
            else {}
        )

        streams = payload.get("streams", [])
        stream = streams[0] if streams else {}
        format_info = payload.get("format", {})

        duration_raw = (
            stream.get("duration")
            or format_info.get("duration")
        )

        try:
            duration = float(duration_raw)
        except (TypeError, ValueError):
            duration = np.nan

        return {
            "returncode": completed.returncode,
            "codec": stream.get("codec_name"),
            "width": stream.get("width"),
            "height": stream.get("height"),
            "r_frame_rate": parse_fraction(
                stream.get("r_frame_rate")
            ),
            "avg_frame_rate": parse_fraction(
                stream.get("avg_frame_rate")
            ),
            "nb_frames": stream.get("nb_frames"),
            "duration_seconds": duration,
            "stderr": completed.stderr[-2000:],
        }


    probe_rows = []

    for row in selected_recordings.itertuples(
        index=False
    ):
        path = Path(row.video_local_path)
        probe = ffprobe_video(path)

        probe_rows.append({
            "recording_name": row.recording_name,
            "video_local_path": str(path),
            **probe,
        })

    batch_probe = pd.DataFrame(probe_rows)

    if not batch_probe.empty:
        if not (
            batch_probe["returncode"] == 0
        ).all():
            display(batch_probe)
            raise RuntimeError(
                "At least one raw recording failed FFprobe."
            )

        if batch_probe[
            "duration_seconds"
        ].isna().any():
            display(batch_probe)
            raise RuntimeError(
                "At least one recording has no duration."
            )

    crop_validation = selected_sequences.merge(
        batch_probe[
            [
                "recording_name",
                "duration_seconds",
                "r_frame_rate",
                "avg_frame_rate",
                "width",
                "height",
                "codec",
            ]
        ],
        on="recording_name",
        how="left",
        validate="many_to_one",
    )

    crop_validation["crop_valid"] = (
        (crop_validation["clip_start_seconds"] >= 0)
        & (
            crop_validation["clip_end_seconds"]
            > crop_validation["clip_start_seconds"]
        )
        & (
            crop_validation["clip_end_seconds"]
            <= crop_validation["duration_seconds"]
            + 1.0
        )
    )

    display(batch_probe)
    display(
        crop_validation[
            [
                "sequence_id",
                "recording_name",
                "clip_start_seconds",
                "clip_end_seconds",
                "duration_seconds",
                "r_frame_rate",
                "crop_valid",
            ]
        ]
    )

    if not crop_validation.empty and not (
        crop_validation["crop_valid"]
    ).all():
        raise RuntimeError(
            "At least one selected annotation crop "
            "falls outside its raw recording."
        )

    batch_probe.to_csv(
        BATCH_ROOT / "batch_ffprobe.csv",
        index=False,
    )
    crop_validation.to_csv(
        BATCH_ROOT / "batch_crop_validation.csv",
        index=False,
    )

    # ===== Validated V4 code cell 29 =====

    # ProcedureVRL's Howto100m_develop loader accepts:
    #
    #   3 fields: path label duration
    #   5 fields: path label duration start end
    #
    # We need the five-field form because each Assembly101 assembly/disassembly
    # sequence is a temporal interval inside a longer raw recording.

    duration_by_recording = (
        batch_probe[
            [
                "recording_name",
                "duration_seconds",
            ]
        ]
        .drop_duplicates(
            subset=["recording_name"]
        )
    )

    sequence_csv_source = selected_sequences.merge(
        duration_by_recording,
        on="recording_name",
        how="left",
        validate="many_to_one",
    )

    if sequence_csv_source["duration_seconds"].isna().any():
        display(
            sequence_csv_source.loc[
                sequence_csv_source[
                    "duration_seconds"
                ].isna(),
                [
                    "sequence_id",
                    "recording_name",
                    "video_local_path",
                ],
            ]
        )
        raise RuntimeError(
            "Missing FFprobe duration for at least one "
            "ProcedureVRL CSV row."
        )

    csv_rows = []
    csv_manifest_rows = []

    for row in sequence_csv_source.itertuples(
        index=False
    ):
        video_path = Path(row.video_local_path)

        if not video_path.exists():
            raise FileNotFoundError(video_path)

        relative_path = video_path.relative_to(
            ASSEMBLY_ROOT
        )
        relative_no_extension = str(
            relative_path.with_suffix("")
        )

        full_video_duration = float(
            row.duration_seconds
        )
        start_seconds = float(
            row.clip_start_seconds
        )
        end_seconds = float(
            row.clip_end_seconds
        )

        if not (
            0.0
            <= start_seconds
            < end_seconds
            <= full_video_duration + 1.0
        ):
            raise ValueError(
                f"Invalid temporal interval for "
                f"{row.sequence_id}: "
                f"start={start_seconds}, "
                f"end={end_seconds}, "
                f"duration={full_video_duration}"
            )

        # The label is unused during hidden-feature extraction, but the
        # ProcedureVRL loader requires an integer label in field 2.
        dummy_integer_label = 0

        # Exact required schema:
        # path label full_video_duration clip_start clip_end
        csv_line = (
            f"{relative_no_extension} "
            f"{dummy_integer_label:d} "
            f"{full_video_duration:.6f} "
            f"{start_seconds:.6f} "
            f"{end_seconds:.6f}"
        )

        if len(csv_line.split(" ")) != 5:
            raise RuntimeError(
                "Malformed five-field ProcedureVRL CSV row: "
                + repr(csv_line)
            )

        csv_rows.append(csv_line)

        csv_manifest_rows.append({
            "run_index": int(row.run_index),
            "sequence_id": row.sequence_id,
            "recording_name": row.recording_name,
            "relative_no_extension": (
                relative_no_extension
            ),
            "dummy_integer_label": (
                dummy_integer_label
            ),
            "full_video_duration": (
                full_video_duration
            ),
            "start_seconds": start_seconds,
            "end_seconds": end_seconds,
            "num_fields": 5,
        })

    if not csv_rows:
        raise RuntimeError(
            "No ProcedureVRL CSV rows were generated."
        )

    test_csv_path = CSV_DIR / "test.csv"
    train_csv_path = CSV_DIR / "train.csv"
    val_csv_path = CSV_DIR / "val.csv"

    test_csv_path.write_text(
        "\n".join(csv_rows) + "\n",
        encoding="utf-8",
    )

    # The loader requires these files to exist even though training and validation
    # are disabled. A valid five-field row is safer than a placeholder.
    train_csv_path.write_text(
        csv_rows[0] + "\n",
        encoding="utf-8",
    )
    val_csv_path.write_text(
        csv_rows[0] + "\n",
        encoding="utf-8",
    )

    # Remove loader path maps generated for an earlier CSV. They cache path
    # resolution by row index and must be rebuilt whenever the CSV changes.
    removed_path_maps = []

    for split_name in ["train", "val", "test"]:
        path_map = (
            CSV_DIR
            / f"path_map_dict_{split_name}.pth"
        )

        if path_map.exists():
            path_map.unlink()
            removed_path_maps.append(
                str(path_map)
            )

    procedurevrl_csv_manifest = pd.DataFrame(
        csv_manifest_rows
    )
    procedurevrl_csv_manifest_path = (
        BATCH_ROOT
        / "procedurevrl_csv_manifest.csv"
    )
    procedurevrl_csv_manifest.to_csv(
        procedurevrl_csv_manifest_path,
        index=False,
    )

    # Parse the written file independently before starting the expensive model.
    written_rows = [
        line.strip()
        for line in test_csv_path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    schema_validation_rows = []

    for row_index, line in enumerate(
        written_rows
    ):
        fields = line.split(" ")

        if len(fields) != 5:
            raise RuntimeError(
                f"CSV row {row_index} has "
                f"{len(fields)} fields, expected 5: "
                f"{line!r}"
            )

        path_field, label_field, duration_field, start_field, end_field = fields

        try:
            parsed_label = int(label_field)
            parsed_duration = float(
                duration_field
            )
            parsed_start = float(start_field)
            parsed_end = float(end_field)
        except ValueError as exc:
            raise RuntimeError(
                f"CSV row {row_index} has invalid "
                f"numeric fields: {line!r}"
            ) from exc

        schema_validation_rows.append({
            "row_index": row_index,
            "path": path_field,
            "label": parsed_label,
            "duration": parsed_duration,
            "start": parsed_start,
            "end": parsed_end,
            "valid_interval": (
                0.0
                <= parsed_start
                < parsed_end
                <= parsed_duration + 1.0
            ),
        })

    schema_validation = pd.DataFrame(
        schema_validation_rows
    )

    if not schema_validation[
        "valid_interval"
    ].all():
        display(
            schema_validation.loc[
                ~schema_validation[
                    "valid_interval"
                ]
            ]
        )
        raise RuntimeError(
            "At least one written CSV interval is invalid."
        )

    print("ProcedureVRL CSV schema: path label duration start end")
    print("ProcedureVRL test rows:", len(written_rows))
    print("All rows have five fields:", True)
    print("Removed stale path maps:", removed_path_maps)
    print("CSV:", test_csv_path)
    print("Manifest:", procedurevrl_csv_manifest_path)
    print("\nFirst rows:")
    print("\n".join(written_rows[:10]))

    display(
        schema_validation.head(20)
    )

    if len(written_rows) != len(
        selected_sequences
    ):
        raise RuntimeError(
            "ProcedureVRL CSV row count mismatch: "
            f"{len(written_rows)} != "
            f"{len(selected_sequences)}"
        )

    # ===== Validated V4 code cell 33 =====
    if selected_sequences.empty:
        print(
            "No selected sequences; skipping ProcedureVRL."
        )
        procedurevrl_hidden = np.empty(
            (
                0,
                NUM_TEMPORAL_STEPS,
                EXPECTED_PROCEDUREVRL_DIM,
            ),
            dtype=np.float32,
        )
        procedurevrl_hook_summary = {}

    else:
        command = [
            sys.executable,
            "tools/feat_extract_hidden_streaming.py",
            "--cfg",
            (
                "configs/HowTo100M/"
                "procedurevrl_adamw.yaml"
            ),

            "TRAIN.ENABLE",
            "False",
            "TRAIN.TEXT",
            "''",
            "DEV.ORDER_PRETRAIN_ENABLED",
            "False",

            "TEST.ENABLE",
            "True",
            "TEST.DATASET",
            "howto100m_develop",
            "TEST.CHECKPOINT_FILE_PATH",
            str(PROCEDUREVRL_CKPT_PATH),
            "TEST.SAVE_PREDICT_PATH",
            str(BATCH_HIDDEN_NPY),
            "TEST.NUM_ENSEMBLE_VIEWS",
            str(
                PROCEDUREVRL_NUM_ENSEMBLE_VIEWS
            ),
            "TEST.NUM_SPATIAL_CROPS",
            "1",
            "TEST.BATCH_SIZE",
            str(PROCEDUREVRL_BATCH_SIZE),

            "DATA.PATH_TO_DATA_DIR",
            str(CSV_DIR),
            "DATA.PATH_PREFIX",
            str(ASSEMBLY_ROOT),
            "DATA.PATH_LABEL_SEPARATOR",
            " ",
            "DATA.NUM_FRAMES",
            "16",
            "DATA.SAMPLING_RATE",
            "6",
            "DATA.FD",
            "3.0",
            "DATA.TEST_CROP_SIZE",
            "224",
            "DATA.TRAIN_JITTER_SCALES",
            "[256,320]",
            "DATA.INPUT_CHANNEL_NUM",
            "[3]",
            "DATA.DECODING_BACKEND",
            "ffmpeg",

            "MODEL.NUM_CLASSES",
            "9871",
            "MODEL.HEAD_ACT",
            "softmax",

            "DATA_LOADER.NUM_WORKERS",
            str(PROCEDUREVRL_NUM_WORKERS),
            "DATA_LOADER.PIN_MEMORY",
            "True",

            "NUM_GPUS",
            "1",
            "NUM_SHARDS",
            "1",
            "OUTPUT_DIR",
            str(PROC_OUTPUT_DIR),
            "LOG_MODEL_INFO",
            "False",
        ]

        overrides = command[4:]

        print("Number of override arguments:", len(overrides))
        print("Override count even:", len(overrides) % 2 == 0)

        if len(overrides) % 2 != 0:
            raise RuntimeError(
                "ProcedureVRL override list is malformed."
            )

        print("Command:")
        print(" ".join(command))

        extraction_start = time.time()

        process = subprocess.run(
            command,
            cwd=str(PROCEDUREVRL_REPO),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            env={
                **os.environ,
                "PYTHONPATH": os.environ.get(
                    "PYTHONPATH",
                    "",
                ),
            },
        )

        BATCH_LOG_PATH.write_text(
            process.stdout,
            encoding="utf-8",
        )

        elapsed_minutes = (
            time.time() - extraction_start
        ) / 60.0

        print("Return code:", process.returncode)
        print("Elapsed minutes:", elapsed_minutes)
        print("Log:", BATCH_LOG_PATH)
        print("\nLast 180 log lines:")
        print(
            "\n".join(
                process.stdout.splitlines()[-180:]
            )
        )

        if process.returncode != 0:
            raise RuntimeError(
                "ProcedureVRL extraction failed. "
                "Inspect the printed log."
            )

        if not BATCH_HIDDEN_NPY.exists():
            raise FileNotFoundError(
                BATCH_HIDDEN_NPY
            )

        procedurevrl_hidden = np.load(
            BATCH_HIDDEN_NPY
        )

        hook_summary_path = (
            BATCH_HIDDEN_NPY.with_suffix(
                ".summary.json"
            )
        )

        procedurevrl_hook_summary = (
            json.loads(
                hook_summary_path.read_text(
                    encoding="utf-8"
                )
            )
            if hook_summary_path.exists()
            else {}
        )

    expected_proc_shape = (
        len(selected_sequences),
        NUM_TEMPORAL_STEPS,
        EXPECTED_PROCEDUREVRL_DIM,
    )

    print(
        "ProcedureVRL raw hidden shape:",
        procedurevrl_hidden.shape,
    )
    print(
        "Expected shape:",
        expected_proc_shape,
    )
    print(
        "Finite:",
        bool(np.isfinite(
            procedurevrl_hidden
        ).all()),
    )

    if procedurevrl_hidden.shape != expected_proc_shape:
        raise RuntimeError(
            "Unexpected ProcedureVRL hidden shape: "
            f"{procedurevrl_hidden.shape} "
            f"!= {expected_proc_shape}"
        )

    if not np.isfinite(
        procedurevrl_hidden
    ).all():
        raise RuntimeError(
            "ProcedureVRL output contains NaN/Inf."
        )

    # ===== Validated V4 code cell 39 =====
    from decord import VideoReader, cpu
    from PIL import Image


    def normalized_center_positions(
        num_steps,
    ):
        return (
            np.arange(num_steps, dtype=np.float64)
            + 0.5
        ) / num_steps


    CENTER_POSITIONS = normalized_center_positions(
        NUM_TEMPORAL_STEPS
    )


    def sample_sequence_frames(
        video_path,
        clip_start_seconds,
        clip_end_seconds,
    ):
        video_reader = VideoReader(
            str(video_path),
            ctx=cpu(0),
        )

        total_frames = len(video_reader)
        fps = float(
            video_reader.get_avg_fps()
        )

        if total_frames <= 0 or fps <= 0:
            raise RuntimeError(
                f"Invalid video metadata for {video_path}: "
                f"frames={total_frames}, fps={fps}"
            )

        absolute_times = (
            float(clip_start_seconds)
            + CENTER_POSITIONS
            * (
                float(clip_end_seconds)
                - float(clip_start_seconds)
            )
        )

        frame_indices = np.rint(
            absolute_times * fps
        ).astype(np.int64)

        frame_indices = np.clip(
            frame_indices,
            0,
            total_frames - 1,
        )

        frames = video_reader.get_batch(
            frame_indices.tolist()
        ).asnumpy()

        return {
            "frames": frames,
            "frame_indices": frame_indices,
            "absolute_times": absolute_times,
            "video_fps": fps,
            "total_frames": total_frames,
        }


    clip_feature_rows = []
    sampling_rows = []

    for row in tqdm(
        selected_sequences.itertuples(index=False),
        total=len(selected_sequences),
        desc="CLIP sequence features",
    ):
        video_path = Path(row.video_local_path)

        sampled = sample_sequence_frames(
            video_path=video_path,
            clip_start_seconds=(
                row.clip_start_seconds
            ),
            clip_end_seconds=(
                row.clip_end_seconds
            ),
        )

        frame_tensors = torch.stack([
            clip_preprocess(
                Image.fromarray(frame)
            )
            for frame in sampled["frames"]
        ])

        embedding_batches = []

        with torch.no_grad():
            for start in range(
                0,
                len(frame_tensors),
                CLIP_FRAME_BATCH_SIZE,
            ):
                batch = frame_tensors[
                    start:
                    start + CLIP_FRAME_BATCH_SIZE
                ].to(CLIP_DEVICE)

                embeddings = clip_model.encode_image(
                    batch
                ).float()

                if CLIP_NORMALIZE_FEATURES:
                    embeddings = (
                        embeddings
                        / embeddings.norm(
                            dim=-1,
                            keepdim=True,
                        ).clamp_min(1e-12)
                    )

                embedding_batches.append(
                    embeddings.cpu()
                )

        clip_sequence_embeddings = torch.cat(
            embedding_batches,
            dim=0,
        ).numpy().astype(np.float32)

        if clip_sequence_embeddings.shape != (
            NUM_TEMPORAL_STEPS,
            EXPECTED_CLIP_DIM,
        ):
            raise RuntimeError(
                "Unexpected CLIP sequence shape for "
                f"{row.sequence_id}: "
                f"{clip_sequence_embeddings.shape}"
            )

        if not np.isfinite(
            clip_sequence_embeddings
        ).all():
            raise RuntimeError(
                "CLIP output contains NaN/Inf for "
                f"{row.sequence_id}"
            )

        output_path = (
            CLIP_FEATURE_DIR
            / f"{row.sequence_id}.npy"
        )

        # [T, D] -> MS-TCN [D, T]
        np.save(
            output_path,
            clip_sequence_embeddings.T,
        )

        clip_feature_rows.append({
            "sequence_id": row.sequence_id,
            "recording_name": row.recording_name,
            "split": row.split,
            "activity": row.activity,
            "feature_path": str(output_path),
            "feature_dim": EXPECTED_CLIP_DIM,
            "feature_len": NUM_TEMPORAL_STEPS,
            "normalized": CLIP_NORMALIZE_FEATURES,
            "clip_model": CLIP_MODEL_NAME,
        })

        for step_index in range(
            NUM_TEMPORAL_STEPS
        ):
            sampling_rows.append({
                "sequence_id": row.sequence_id,
                "recording_name": row.recording_name,
                "step_index": step_index,
                "normalized_center": float(
                    CENTER_POSITIONS[step_index]
                ),
                "absolute_time_seconds": float(
                    sampled["absolute_times"][
                        step_index
                    ]
                ),
                "raw_video_frame_index": int(
                    sampled["frame_indices"][
                        step_index
                    ]
                ),
                "raw_video_fps": float(
                    sampled["video_fps"]
                ),
                "raw_video_total_frames": int(
                    sampled["total_frames"]
                ),
            })

    batch_clip_feature_manifest = pd.DataFrame(
        clip_feature_rows
    )
    batch_sampling_manifest = pd.DataFrame(
        sampling_rows
    )

    batch_clip_feature_manifest.to_csv(
        BATCH_ROOT
        / "batch_clip_feature_manifest.csv",
        index=False,
    )
    batch_sampling_manifest.to_csv(
        BATCH_ROOT
        / "batch_sampling_manifest.csv",
        index=False,
    )

    display(batch_clip_feature_manifest.head())
    display(batch_sampling_manifest.head(32))

    print(
        "CLIP feature files written:",
        len(batch_clip_feature_manifest),
    )

    # ===== Validated V4 code cell 41 =====
    procedurevrl_feature_rows = []

    for index, row in selected_sequences.iterrows():
        feature = (
            procedurevrl_hidden[
                index
            ]
            .astype(np.float32)
            .T
        )

        if feature.shape != (
            EXPECTED_PROCEDUREVRL_DIM,
            NUM_TEMPORAL_STEPS,
        ):
            raise RuntimeError(
                f"Unexpected ProcedureVRL feature shape "
                f"for {row['sequence_id']}: "
                f"{feature.shape}"
            )

        output_path = (
            PROC_FEATURE_DIR
            / f"{row['sequence_id']}.npy"
        )
        np.save(output_path, feature)

        procedurevrl_feature_rows.append({
            "sequence_id": row["sequence_id"],
            "recording_name": row[
                "recording_name"
            ],
            "split": row["split"],
            "activity": row["activity"],
            "feature_path": str(output_path),
            "feature_dim": int(feature.shape[0]),
            "feature_len": int(feature.shape[1]),
            "hook_target": (
                procedurevrl_hook_summary
                .get("hook_target", {})
                .get("name")
            ),
        })

    batch_procedurevrl_feature_manifest = (
        pd.DataFrame(
            procedurevrl_feature_rows
        )
    )

    batch_procedurevrl_feature_manifest.to_csv(
        BATCH_ROOT
        / "batch_procedurevrl_feature_manifest.csv",
        index=False,
    )

    display(
        batch_procedurevrl_feature_manifest.head()
    )

    print(
        "ProcedureVRL feature files written:",
        len(
            batch_procedurevrl_feature_manifest
        ),
    )

    # ===== Validated V4 code cell 43 =====
    alignment_rows = []


    def label_at_normalized_centers(
        dense_labels,
        num_steps,
    ):
        dense_labels = list(dense_labels)

        if not dense_labels:
            raise ValueError(
                "Dense ground truth is empty."
            )

        centers = normalized_center_positions(
            num_steps
        )

        dense_indices = np.floor(
            centers * len(dense_labels)
        ).astype(np.int64)

        dense_indices = np.clip(
            dense_indices,
            0,
            len(dense_labels) - 1,
        )

        sampled_labels = [
            dense_labels[int(index)]
            for index in dense_indices
        ]

        return sampled_labels, dense_indices


    for row in selected_sequences.itertuples(
        index=False
    ):
        dense_gt_path = (
            SOURCE_DENSE_GT_DIR
            / f"{row.sequence_id}.txt"
        )

        dense_labels = read_nonempty_lines(
            dense_gt_path
        )

        expected_dense_length = int(
            row.ground_truth_length_30fps
        )

        if len(dense_labels) != expected_dense_length:
            raise RuntimeError(
                f"Dense GT length mismatch for "
                f"{row.sequence_id}: "
                f"{len(dense_labels)} != "
                f"{expected_dense_length}"
            )

        sampled_labels, dense_indices = (
            label_at_normalized_centers(
                dense_labels,
                NUM_TEMPORAL_STEPS,
            )
        )

        unknown_labels = sorted(
            set(sampled_labels) - MAPPING_LABELS
        )

        if unknown_labels:
            raise RuntimeError(
                f"Unknown sampled labels for "
                f"{row.sequence_id}: "
                f"{unknown_labels}"
            )

        proc_gt_path = (
            PROC_GT_DIR
            / f"{row.sequence_id}.txt"
        )
        clip_gt_path = (
            CLIP_GT_DIR
            / f"{row.sequence_id}.txt"
        )

        gt_text = (
            "\n".join(sampled_labels)
            + "\n"
        )

        proc_gt_path.write_text(
            gt_text,
            encoding="utf-8",
        )
        clip_gt_path.write_text(
            gt_text,
            encoding="utf-8",
        )

        for step_index, (
            dense_index,
            label,
        ) in enumerate(
            zip(
                dense_indices,
                sampled_labels,
            )
        ):
            normalized_center = float(
                CENTER_POSITIONS[step_index]
            )

            alignment_rows.append({
                "sequence_id": row.sequence_id,
                "recording_name": row.recording_name,
                "split": row.split,
                "activity": row.activity,
                "step_index": step_index,
                "normalized_center": (
                    normalized_center
                ),
                "dense_gt_index_30fps": int(
                    dense_index
                ),
                "dense_gt_length_30fps": len(
                    dense_labels
                ),
                "approximate_absolute_time_seconds": (
                    float(row.clip_start_seconds)
                    + normalized_center
                    * float(
                        row.clip_duration_seconds
                    )
                ),
                "action_label": label,
            })

    batch_alignment = pd.DataFrame(
        alignment_rows
    )
    batch_alignment.to_csv(
        BATCH_ALIGNMENT_PATH,
        index=False,
    )

    # Update the persistent alignment manifest without duplicates.
    if TEMPORAL_ALIGNMENT_PATH.exists():
        old_alignment = pd.read_csv(
            TEMPORAL_ALIGNMENT_PATH
        )
        old_alignment = old_alignment[
            ~old_alignment["sequence_id"].isin(
                set(selected_sequences[
                    "sequence_id"
                ])
            )
        ]

        all_alignment = pd.concat(
            [
                old_alignment,
                batch_alignment,
            ],
            ignore_index=True,
        )
    else:
        all_alignment = batch_alignment.copy()

    all_alignment = all_alignment.sort_values(
        [
            "sequence_id",
            "step_index",
        ]
    ).reset_index(drop=True)

    all_alignment.to_csv(
        TEMPORAL_ALIGNMENT_PATH,
        index=False,
    )

    print(
        "Batch aligned GT rows:",
        len(batch_alignment),
    )
    print(
        "Persistent alignment rows:",
        len(all_alignment),
    )
    print("Saved:", TEMPORAL_ALIGNMENT_PATH)

    display(batch_alignment.head(32))

    # ===== Validated V4 code cell 45 =====
    batch_validation_rows = []

    for row in selected_sequences.itertuples(
        index=False
    ):
        proc_path = (
            PROC_FEATURE_DIR
            / f"{row.sequence_id}.npy"
        )
        clip_path = (
            CLIP_FEATURE_DIR
            / f"{row.sequence_id}.npy"
        )
        proc_gt_path = (
            PROC_GT_DIR
            / f"{row.sequence_id}.txt"
        )
        clip_gt_path = (
            CLIP_GT_DIR
            / f"{row.sequence_id}.txt"
        )

        proc = inspect_feature(
            proc_path,
            EXPECTED_PROCEDUREVRL_DIM,
        )
        clip_result = inspect_feature(
            clip_path,
            EXPECTED_CLIP_DIM,
        )
        proc_gt = inspect_coarse_gt(
            proc_gt_path
        )
        clip_gt = inspect_coarse_gt(
            clip_gt_path
        )

        gt_equal = bool(
            proc_gt["valid"]
            and clip_gt["valid"]
            and (
                read_nonempty_lines(proc_gt_path)
                == read_nonempty_lines(
                    clip_gt_path
                )
            )
        )

        complete = bool(
            proc["valid"]
            and clip_result["valid"]
            and proc_gt["valid"]
            and clip_gt["valid"]
            and gt_equal
        )

        batch_validation_rows.append({
            "sequence_id": row.sequence_id,
            "recording_name": row.recording_name,
            "procedurevrl_valid": proc["valid"],
            "procedurevrl_shape": proc["shape"],
            "clip_valid": clip_result["valid"],
            "clip_shape": clip_result["shape"],
            "procedurevrl_gt_valid": proc_gt["valid"],
            "clip_gt_valid": clip_gt["valid"],
            "gt_equal": gt_equal,
            "complete_all": complete,
        })

    batch_validation = pd.DataFrame(
        batch_validation_rows
    )
    batch_validation.to_csv(
        BATCH_ROOT
        / "batch_feature_validation.csv",
        index=False,
    )

    display(batch_validation)

    if not batch_validation.empty and not (
        batch_validation["complete_all"]
    ).all():
        raise RuntimeError(
            "At least one selected sequence failed "
            "feature/GT validation. Raw videos will "
            "not be deleted."
        )

    print(
        "Selected sequences fully valid:",
        int(
            batch_validation[
                "complete_all"
            ].sum()
        ),
        "/",
        len(batch_validation),
    )

    # ===== Validated V4 code cell 47 =====
    sequence_status = rebuild_sequence_status()
    recording_status = rebuild_recording_status(
        sequence_status
    )

    complete_sequence_ids = set(
        sequence_status.loc[
            sequence_status["complete_all"],
            "sequence_id",
        ]
    )

    for split_name in [
        "train",
        "val",
        "test",
    ]:
        split_sequence_ids = (
            sequences.loc[
                sequences["split"] == split_name,
                "sequence_id",
            ]
            .astype(str)
            .tolist()
        )

        complete_split_ids = [
            sequence_id
            for sequence_id in split_sequence_ids
            if sequence_id in complete_sequence_ids
        ]

        partial_lines = [
            f"{sequence_id}.txt"
            for sequence_id in complete_split_ids
        ]

        for split_dir in [
            PROC_SPLIT_DIR,
            CLIP_SPLIT_DIR,
        ]:
            (
                split_dir
                / f"{split_name}.partial.bundle"
            ).write_text(
                "\n".join(partial_lines)
                + (
                    "\n"
                    if partial_lines
                    else ""
                ),
                encoding="utf-8",
            )

    all_sequences_complete = bool(
        sequence_status["complete_all"].all()
    )

    if all_sequences_complete:
        for source_path in SOURCE_SPLIT_DIR.glob(
            "*.bundle"
        ):
            for target_dir in [
                PROC_SPLIT_DIR,
                CLIP_SPLIT_DIR,
            ]:
                shutil.copy2(
                    source_path,
                    target_dir / source_path.name,
                )

    print(
        "Global complete sequences:",
        int(sequence_status["complete_all"].sum()),
        "/",
        len(sequence_status),
    )
    print(
        "Global complete recordings:",
        int(recording_status["complete_all"].sum()),
        "/",
        len(recording_status),
    )
    print(
        "Final split bundles created:",
        all_sequences_complete,
    )

    display(
        sequence_status[
            [
                "split",
                "complete_all",
            ]
        ]
        .groupby("split")
        .agg(
            complete=(
                "complete_all",
                "sum",
            ),
            total=(
                "complete_all",
                "size",
            ),
        )
        .reset_index()
    )

    # ===== Validated V4 code cell 49 =====
    deletion_rows = []

    if not DELETE_RAW_AFTER_SUCCESS:
        print(
            "Raw deletion disabled in this run."
        )

    else:
        # Delete every locally present raw video whose recording is
        # globally complete. This also cleans up successful smoke videos.
        deletable = recording_status[
            recording_status["complete_all"]
            & recording_status["raw_video_exists"]
        ].copy()

        for row in deletable.itertuples(
            index=False
        ):
            path = Path(row.video_local_path)

            allowed_root = (
                ASSEMBLY_ROOT / "recordings"
            ).resolve()

            try:
                resolved = path.resolve()

                if os.path.commonpath(
                    [
                        str(allowed_root),
                        str(resolved),
                    ]
                ) != str(allowed_root):
                    raise RuntimeError(
                        "Refusing to delete path outside "
                        f"{allowed_root}: {resolved}"
                    )

                if path.name != "C10095_rgb.mp4":
                    raise RuntimeError(
                        "Refusing to delete unexpected "
                        f"filename: {path.name}"
                    )

                size_before = (
                    path.stat().st_size
                    if path.exists()
                    else 0
                )

                if path.exists():
                    path.unlink()

                deleted = not path.exists()

                # Remove an empty per-recording directory only.
                parent_removed = False

                if (
                    deleted
                    and path.parent.exists()
                    and not any(
                        path.parent.iterdir()
                    )
                ):
                    path.parent.rmdir()
                    parent_removed = True

                deletion_rows.append({
                    "recording_name": row.recording_name,
                    "video_path": str(path),
                    "size_bytes_freed": size_before,
                    "deleted": deleted,
                    "empty_parent_removed": (
                        parent_removed
                    ),
                    "error": "",
                })

            except Exception as exc:
                deletion_rows.append({
                    "recording_name": row.recording_name,
                    "video_path": str(path),
                    "size_bytes_freed": 0,
                    "deleted": False,
                    "empty_parent_removed": False,
                    "error": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                })

    deletion_manifest = pd.DataFrame(
        deletion_rows
    )
    deletion_manifest_path = (
        BATCH_ROOT
        / "raw_video_deletion_manifest.csv"
    )
    deletion_manifest.to_csv(
        deletion_manifest_path,
        index=False,
    )

    display(deletion_manifest)

    if not deletion_manifest.empty:
        failed_deletions = deletion_manifest[
            ~deletion_manifest["deleted"]
        ]

        print(
            "Raw recordings deleted:",
            int(deletion_manifest["deleted"].sum()),
            "/",
            len(deletion_manifest),
        )
        print(
            "Space freed:",
            f"{deletion_manifest['size_bytes_freed'].sum() / 1024**3:.3f} GiB",
        )

        if not failed_deletions.empty:
            print(
                "WARNING: some eligible raw videos "
                "were not deleted."
            )

    print("Deletion manifest:", deletion_manifest_path)

    # Refresh raw-file state after deletion.
    recording_status = rebuild_recording_status(
        sequence_status
    )

    # ===== Validated V4 code cell 51 =====
    global_complete_sequences = int(
        sequence_status["complete_all"].sum()
    )
    global_complete_recordings = int(
        recording_status["complete_all"].sum()
    )

    batch_complete_sequences = int(
        batch_validation["complete_all"].sum()
    ) if not batch_validation.empty else 0

    deleted_count = int(
        deletion_manifest["deleted"].sum()
    ) if not deletion_manifest.empty else 0

    freed_bytes = int(
        deletion_manifest["size_bytes_freed"].sum()
    ) if not deletion_manifest.empty else 0

    batch_summary = {
        "status": "completed",
        "run_mode": RUN_MODE,
        "batch_name": BATCH_NAME,
        "selected_recordings": int(
            len(selected_recordings)
        ),
        "selected_sequences": int(
            len(selected_sequences)
        ),
        "selected_sequences_valid": (
            batch_complete_sequences
        ),
        "procedurevrl": {
            "representation": (
                "hidden embedding captured before "
                "the selected final classification head"
            ),
            "feature_shape": [
                EXPECTED_PROCEDUREVRL_DIM,
                NUM_TEMPORAL_STEPS,
            ],
            "num_ensemble_views": (
                PROCEDUREVRL_NUM_ENSEMBLE_VIEWS
            ),
            "hook_summary": (
                procedurevrl_hook_summary
            ),
            "checkpoint": str(
                PROCEDUREVRL_CKPT_PATH
            ),
            "git_commit": git_commit,
        },
        "clip": {
            "model": CLIP_MODEL_NAME,
            "feature_shape": [
                EXPECTED_CLIP_DIM,
                NUM_TEMPORAL_STEPS,
            ],
            "normalized": (
                CLIP_NORMALIZE_FEATURES
            ),
            "text_embedding_shape": list(
                text_embeddings_np.shape
            ),
            "text_prompt_template": (
                TEXT_PROMPT_TEMPLATE
            ),
        },
        "temporal_alignment": {
            "num_steps": NUM_TEMPORAL_STEPS,
            "positions": (
                "uniform normalized interval centers"
            ),
            "annotation_fps": ANNOTATION_FPS,
            "note": (
                "CLIP frames use exact center timestamps. "
                "ProcedureVRL labels use the same normalized "
                "centers as an explicit approximation because "
                "the original extractor does not export its "
                "internal crop timestamps."
            ),
        },
        "raw_cleanup": {
            "enabled": DELETE_RAW_AFTER_SUCCESS,
            "recordings_deleted": deleted_count,
            "space_freed_gib": (
                freed_bytes / 1024**3
            ),
        },
        "global_progress_after_batch": {
            "complete_sequences": (
                global_complete_sequences
            ),
            "total_sequences": int(
                len(sequence_status)
            ),
            "complete_recordings": (
                global_complete_recordings
            ),
            "total_recordings": int(
                len(recording_status)
            ),
            "all_complete": bool(
                sequence_status["complete_all"].all()
            ),
        },
        "paths": {
            "batch_root": str(BATCH_ROOT),
            "procedurevrl_dataset_root": str(
                PROC_DATASET_ROOT
            ),
            "clip_dataset_root": str(
                CLIP_DATASET_ROOT
            ),
            "sequence_status": str(
                SEQUENCE_STATUS_PATH
            ),
            "recording_status": str(
                RECORDING_STATUS_PATH
            ),
            "temporal_alignment": str(
                TEMPORAL_ALIGNMENT_PATH
            ),
            "procedurevrl_log": str(
                BATCH_LOG_PATH
            ),
        },
    }

    BATCH_SUMMARY_PATH.write_text(
        json.dumps(
            batch_summary,
            indent=2,
        ),
        encoding="utf-8",
    )

    global_summary = {
        "status": (
            "completed_all_features"
            if sequence_status[
                "complete_all"
            ].all()
            else "in_progress"
        ),
        "dataset": "Assembly101",
        "view": "v1",
        "camera_file": "C10095_rgb.mp4",
        "num_sequences_complete": (
            global_complete_sequences
        ),
        "num_sequences_total": int(
            len(sequence_status)
        ),
        "num_recordings_complete": (
            global_complete_recordings
        ),
        "num_recordings_total": int(
            len(recording_status)
        ),
        "procedurevrl_feature_shape": [
            EXPECTED_PROCEDUREVRL_DIM,
            NUM_TEMPORAL_STEPS,
        ],
        "clip_feature_shape": [
            EXPECTED_CLIP_DIM,
            NUM_TEMPORAL_STEPS,
        ],
        "clip_text_embedding_shape": list(
            text_embeddings_np.shape
        ),
        "last_batch": BATCH_NAME,
        "last_updated_utc": (
            datetime.now(
                timezone.utc
            ).isoformat()
        ),
        "next_step": (
            "Run the same notebook again in stream mode."
            if not sequence_status[
                "complete_all"
            ].all()
            else (
                "Train visual-only MS-TCN and LTContext "
                "baselines separately on ProcedureVRL "
                "and CLIP features."
            )
        ),
        "paths": {
            "procedurevrl_dataset_root": str(
                PROC_DATASET_ROOT
            ),
            "clip_dataset_root": str(
                CLIP_DATASET_ROOT
            ),
            "clip_text_embeddings": str(
                clip_text_embedding_path
            ),
            "sequence_status": str(
                SEQUENCE_STATUS_PATH
            ),
            "recording_status": str(
                RECORDING_STATUS_PATH
            ),
            "temporal_alignment": str(
                TEMPORAL_ALIGNMENT_PATH
            ),
        },
    }

    FINAL_SUMMARY_PATH.write_text(
        json.dumps(
            global_summary,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("Batch summary:")
    print(
        json.dumps(
            batch_summary,
            indent=2,
        )
    )

    print("\nGlobal summary:")
    print(
        json.dumps(
            global_summary,
            indent=2,
        )
    )

    print("\nSaved:")
    print(BATCH_SUMMARY_PATH)
    print(FINAL_SUMMARY_PATH)

    if RUN_MODE == "smoke":
        print("\nSmoke run complete.")
        print("For streaming extraction, change:")
        print('RUN_MODE = "stream"')
        print(
            "STREAM_BATCH_RECORDINGS = 5"
        )
        print(
            "DELETE_RAW_AFTER_SUCCESS will "
            "automatically become True."
        )

    elif not sequence_status[
        "complete_all"
    ].all():
        print(
            "\nStreaming extraction is not complete. "
            "Rerun this notebook for the next batch."
        )

    else:
        print(
            "\nAll ProcedureVRL and CLIP features "
            "are complete."
        )
        print("Next notebook:")
        print(
            "16_assembly101_visual_only_mstcn_"
            "baselines_COLAB.ipynb"
        )

    return global_summary



## 13. Run the automatic multi-batch session

Run this cell and leave the Colab tab open. Every completed batch is persisted
to Drive. A failed batch is retried; after the final failed retry the notebook
stops safely and saves the traceback under:

```text
.../streaming_visual_features_v1/logs/
```

A normal partial session ends with `safe_time_limit_reached`. A completely
finished dataset ends with `completed_all_features`.


In [30]:

import gc
import traceback
import uuid

AUTOLOOP_HISTORY_PATH = (
    MANIFEST_ROOT / "autoloop_session_history.csv"
)
AUTOLOOP_LATEST_SUMMARY_PATH = (
    OUT_ROOT / "autoloop_latest_session_summary.json"
)


def append_autoloop_history(row):
    new_row = pd.DataFrame([row])

    if AUTOLOOP_HISTORY_PATH.exists():
        old = pd.read_csv(
            AUTOLOOP_HISTORY_PATH,
            keep_default_na=False,
        )
        history = pd.concat(
            [old, new_row],
            ignore_index=True,
        )
    else:
        history = new_row

    history.to_csv(
        AUTOLOOP_HISTORY_PATH,
        index=False,
    )


def current_progress():
    current_sequence_status = rebuild_sequence_status()
    current_recording_status = rebuild_recording_status(
        current_sequence_status
    )

    return {
        "sequence_status": current_sequence_status,
        "recording_status": current_recording_status,
        "complete_sequences": int(
            current_sequence_status["complete_all"].sum()
        ),
        "total_sequences": int(
            len(current_sequence_status)
        ),
        "complete_recordings": int(
            current_recording_status["complete_all"].sum()
        ),
        "total_recordings": int(
            len(current_recording_status)
        ),
        "all_complete": bool(
            current_sequence_status["complete_all"].all()
        ),
    }


if not AUTO_LOOP:
    raise RuntimeError("AUTO_LOOP must be True.")

if RUN_MODE != "stream":
    raise RuntimeError(
        "AUTOLOOP V5 requires RUN_MODE='stream'."
    )

if not DELETE_RAW_AFTER_SUCCESS:
    raise RuntimeError(
        "Validated raw-video cleanup must be enabled."
    )


session_id = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

session_started_wall = datetime.now(timezone.utc)
session_started_monotonic = time.monotonic()

deadline_monotonic = (
    session_started_monotonic
    + MAX_SESSION_HOURS * 3600
)
safe_stop_monotonic = (
    deadline_monotonic
    - STOP_BUFFER_MINUTES * 60
)

session_successful_batches = 0
session_failed_attempts = 0
session_stop_reason = None
last_batch_summary = None

print("=" * 80)
print("AUTOLOOP SESSION START")
print("=" * 80)
print("session_id:", session_id)
print("batch size:", STREAM_BATCH_RECORDINGS)
print("maximum batches:", MAX_BATCHES_PER_SESSION)
print("maximum session hours:", MAX_SESSION_HOURS)
print("safe stop buffer minutes:", STOP_BUFFER_MINUTES)
print("maximum retries per batch:", MAX_RETRIES_PER_BATCH)
print("history:", AUTOLOOP_HISTORY_PATH)

initial = current_progress()
sequence_status = initial["sequence_status"]
recording_status = initial["recording_status"]

print(
    "initial progress:",
    f"{initial['complete_recordings']}/"
    f"{initial['total_recordings']} recordings;",
    f"{initial['complete_sequences']}/"
    f"{initial['total_sequences']} sequences",
)

for automatic_batch_index in range(
    1,
    MAX_BATCHES_PER_SESSION + 1,
):
    progress_before = current_progress()
    sequence_status = progress_before[
        "sequence_status"
    ]
    recording_status = progress_before[
        "recording_status"
    ]

    if progress_before["all_complete"]:
        session_stop_reason = "completed_all_features"
        print("\nAll Assembly101 features are complete.")
        break

    now_monotonic = time.monotonic()

    if now_monotonic >= safe_stop_monotonic:
        session_stop_reason = "safe_time_limit_reached"
        print(
            "\nStopping before the configured "
            "Colab session deadline."
        )
        break

    elapsed_hours = (
        now_monotonic - session_started_monotonic
    ) / 3600.0
    remaining_safe_minutes = (
        safe_stop_monotonic - now_monotonic
    ) / 60.0

    print("\n" + "=" * 80)
    print(
        f"AUTOMATIC BATCH "
        f"{automatic_batch_index}/"
        f"{MAX_BATCHES_PER_SESSION}"
    )
    print("=" * 80)
    print(
        "progress before:",
        f"{progress_before['complete_recordings']}/"
        f"{progress_before['total_recordings']} recordings;",
        f"{progress_before['complete_sequences']}/"
        f"{progress_before['total_sequences']} sequences",
    )
    print(
        "session elapsed hours:",
        f"{elapsed_hours:.2f}",
    )
    print(
        "safe processing time remaining:",
        f"{remaining_safe_minutes:.1f} minutes",
    )

    batch_succeeded = False

    for retry_index in range(
        1,
        MAX_RETRIES_PER_BATCH + 1,
    ):
        attempt_started_wall = datetime.now(
            timezone.utc
        )
        attempt_started_monotonic = time.monotonic()

        print(
            f"\nAttempt {retry_index}/"
            f"{MAX_RETRIES_PER_BATCH}"
        )

        try:
            last_batch_summary = process_one_stream_batch()
            progress_after = current_progress()

            sequence_status = progress_after[
                "sequence_status"
            ]
            recording_status = progress_after[
                "recording_status"
            ]

            recording_gain = (
                progress_after["complete_recordings"]
                - progress_before["complete_recordings"]
            )
            sequence_gain = (
                progress_after["complete_sequences"]
                - progress_before["complete_sequences"]
            )

            if recording_gain <= 0:
                raise RuntimeError(
                    "The batch finished without increasing "
                    "the number of complete recordings."
                )

            attempt_minutes = (
                time.monotonic()
                - attempt_started_monotonic
            ) / 60.0

            append_autoloop_history(
                {
                    "session_id": session_id,
                    "automatic_batch_index": (
                        automatic_batch_index
                    ),
                    "retry_index": retry_index,
                    "status": "completed",
                    "started_utc": (
                        attempt_started_wall.isoformat()
                    ),
                    "finished_utc": (
                        datetime.now(
                            timezone.utc
                        ).isoformat()
                    ),
                    "duration_minutes": attempt_minutes,
                    "recordings_before": (
                        progress_before[
                            "complete_recordings"
                        ]
                    ),
                    "recordings_after": (
                        progress_after[
                            "complete_recordings"
                        ]
                    ),
                    "recording_gain": recording_gain,
                    "sequences_before": (
                        progress_before[
                            "complete_sequences"
                        ]
                    ),
                    "sequences_after": (
                        progress_after[
                            "complete_sequences"
                        ]
                    ),
                    "sequence_gain": sequence_gain,
                    "batch_name": (
                        last_batch_summary.get(
                            "batch_name",
                            "",
                        )
                    ),
                    "error_log": "",
                }
            )

            session_successful_batches += 1
            batch_succeeded = True

            print("\nAutomatic batch completed.")
            print("recording gain:", recording_gain)
            print("sequence gain:", sequence_gain)
            print(
                "progress after:",
                f"{progress_after['complete_recordings']}/"
                f"{progress_after['total_recordings']} recordings;",
                f"{progress_after['complete_sequences']}/"
                f"{progress_after['total_sequences']} sequences",
            )

            time.sleep(DRIVE_FLUSH_SECONDS)
            gc.collect()
            torch.cuda.empty_cache()
            break

        except Exception as exc:
            session_failed_attempts += 1
            attempt_minutes = (
                time.monotonic()
                - attempt_started_monotonic
            ) / 60.0

            error_text = traceback.format_exc()
            error_log_path = (
                LOG_ROOT
                / (
                    "autoloop_error_"
                    f"{session_id}_"
                    f"batch{automatic_batch_index:03d}_"
                    f"attempt{retry_index:02d}.txt"
                )
            )
            error_log_path.write_text(
                error_text,
                encoding="utf-8",
            )

            failed_progress = current_progress()
            sequence_status = failed_progress[
                "sequence_status"
            ]
            recording_status = failed_progress[
                "recording_status"
            ]

            append_autoloop_history(
                {
                    "session_id": session_id,
                    "automatic_batch_index": (
                        automatic_batch_index
                    ),
                    "retry_index": retry_index,
                    "status": "failed",
                    "started_utc": (
                        attempt_started_wall.isoformat()
                    ),
                    "finished_utc": (
                        datetime.now(
                            timezone.utc
                        ).isoformat()
                    ),
                    "duration_minutes": attempt_minutes,
                    "recordings_before": (
                        progress_before[
                            "complete_recordings"
                        ]
                    ),
                    "recordings_after": (
                        failed_progress[
                            "complete_recordings"
                        ]
                    ),
                    "recording_gain": (
                        failed_progress[
                            "complete_recordings"
                        ]
                        - progress_before[
                            "complete_recordings"
                        ]
                    ),
                    "sequences_before": (
                        progress_before[
                            "complete_sequences"
                        ]
                    ),
                    "sequences_after": (
                        failed_progress[
                            "complete_sequences"
                        ]
                    ),
                    "sequence_gain": (
                        failed_progress[
                            "complete_sequences"
                        ]
                        - progress_before[
                            "complete_sequences"
                        ]
                    ),
                    "batch_name": (
                        last_batch_summary.get(
                            "batch_name",
                            "",
                        )
                        if isinstance(
                            last_batch_summary,
                            dict,
                        )
                        else ""
                    ),
                    "error_log": str(error_log_path),
                }
            )

            print(
                "\nBatch attempt failed:",
                f"{type(exc).__name__}: {exc}",
            )
            print(
                "Full traceback saved:",
                error_log_path,
            )

            gc.collect()
            torch.cuda.empty_cache()

            if retry_index < MAX_RETRIES_PER_BATCH:
                print(
                    "Retrying after a short pause."
                )
                time.sleep(
                    max(
                        RETRY_SLEEP_SECONDS,
                        DRIVE_FLUSH_SECONDS,
                    )
                )
            else:
                print(
                    "No retries remain for this batch."
                )

    if not batch_succeeded:
        session_stop_reason = (
            "unrecoverable_batch_failure"
        )

        if STOP_AFTER_UNRECOVERABLE_BATCH_FAILURE:
            print(
                "\nAUTOLOOP stopped safely after "
                "an unrecoverable batch failure."
            )
            break

if session_stop_reason is None:
    final_check = current_progress()

    if final_check["all_complete"]:
        session_stop_reason = "completed_all_features"
    elif (
        session_successful_batches
        >= MAX_BATCHES_PER_SESSION
    ):
        session_stop_reason = "maximum_batches_reached"
    else:
        session_stop_reason = "loop_finished"

final_progress = current_progress()
sequence_status = final_progress["sequence_status"]
recording_status = final_progress["recording_status"]

session_duration_hours = (
    time.monotonic() - session_started_monotonic
) / 3600.0

session_summary = {
    "session_id": session_id,
    "status": session_stop_reason,
    "started_utc": session_started_wall.isoformat(),
    "finished_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "duration_hours": session_duration_hours,
    "successful_batches": (
        session_successful_batches
    ),
    "failed_attempts": session_failed_attempts,
    "configuration": {
        "stream_batch_recordings": (
            STREAM_BATCH_RECORDINGS
        ),
        "max_batches_per_session": (
            MAX_BATCHES_PER_SESSION
        ),
        "max_session_hours": MAX_SESSION_HOURS,
        "stop_buffer_minutes": (
            STOP_BUFFER_MINUTES
        ),
        "max_retries_per_batch": (
            MAX_RETRIES_PER_BATCH
        ),
        "delete_raw_after_success": (
            DELETE_RAW_AFTER_SUCCESS
        ),
    },
    "final_progress": {
        "complete_recordings": (
            final_progress["complete_recordings"]
        ),
        "total_recordings": (
            final_progress["total_recordings"]
        ),
        "complete_sequences": (
            final_progress["complete_sequences"]
        ),
        "total_sequences": (
            final_progress["total_sequences"]
        ),
        "all_complete": (
            final_progress["all_complete"]
        ),
    },
    "last_batch_summary": last_batch_summary,
    "paths": {
        "global_feature_summary": str(
            FINAL_SUMMARY_PATH
        ),
        "autoloop_history": str(
            AUTOLOOP_HISTORY_PATH
        ),
        "latest_session_summary": str(
            AUTOLOOP_LATEST_SUMMARY_PATH
        ),
        "sequence_status": str(
            SEQUENCE_STATUS_PATH
        ),
        "recording_status": str(
            RECORDING_STATUS_PATH
        ),
        "logs": str(LOG_ROOT),
    },
}

AUTOLOOP_LATEST_SUMMARY_PATH.write_text(
    json.dumps(
        session_summary,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 80)
print("AUTOLOOP SESSION FINISHED")
print("=" * 80)
print(
    json.dumps(
        session_summary,
        indent=2,
    )
)

if final_progress["all_complete"]:
    print(
        "\nAll Assembly101 ProcedureVRL and CLIP "
        "features are complete."
    )
    print(
        "Next notebook: "
        "16_assembly101_visual_only_mstcn_"
        "baselines_COLAB.ipynb"
    )
else:
    print(
        "\nFeature extraction is not complete."
    )
    print(
        "Reconnect and run this same AUTOLOOP V5 "
        "notebook from the first cell."
    )


AUTOLOOP SESSION START
session_id: 20260724T075010Z_133df75e
batch size: 10
maximum batches: 100
maximum session hours: 8.0
safe stop buffer minutes: 45
maximum retries per batch: 2
history: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/manifests/autoloop_session_history.csv
initial progress: 350/350 recordings; 680/680 sequences

All Assembly101 features are complete.

AUTOLOOP SESSION FINISHED
{
  "session_id": "20260724T075010Z_133df75e",
  "status": "completed_all_features",
  "started_utc": "2026-07-24T07:50:10.504847+00:00",
  "finished_utc": "2026-07-24T07:51:27.681283+00:00",
  "duration_hours": 0.0214377733247218,
  "successful_batches": 0,
  "failed_attempts": 0,
  "configuration": {
    "stream_batch_recordings": 10,
    "max_batches_per_session": 100,
    "max_session_hours": 8.0,
    "stop_buffer_minutes": 45,
    "max_retries_per_batch": 2,
    "delete_raw_after_success": true
  },
  "final_progress"


## Usage

1. Select a GPU runtime.
2. Run all cells.
3. Leave the tab running.
4. After a disconnect or `safe_time_limit_reached`, reconnect and run this
   same notebook again from the first cell.
5. Stop only when the summary reports:

```json
{
  "status": "completed_all_features",
  "complete_recordings": 350,
  "total_recordings": 350,
  "complete_sequences": 680,
  "total_sequences": 680,
  "all_complete": true
}
```

After one stable automatic session, `STREAM_BATCH_RECORDINGS` may be increased
from `10` to `20`. Ten is safer for the first run.
